# WTe₂ TIFET ML pipeline: complete standalone notebook
**ECE 455/555 course project, Shuvankar Biswas**

This notebook contains **the entire codebase** as cells: data loading and validation, targets and swing, evaluation protocols with blocked CV, naive baselines, ridge, polynomial ridge, gradient boosting, Gaussian processes, the MLP, PCA and clustering, input-noise robustness, cluster-bootstrap confidence intervals, GP coverage, the epoch-cap check and every figure. **Only the data** (the four NEGF CSV files and `NEGF_all.mat`) is downloaded from GitHub; everything else is computed here.

**How to run (Kaggle recommended):** turn **Internet on** (needed for the data download), choose a CPU machine, then *Run All*. Settings are in the next two cells. A full run uses 4 parallel fits; expect about 1–1.5 h on Kaggle (2 h 16 min with a serial MLP on the author's laptop). Finished jobs are cached under `results/cache`, so re-running after an interruption continues where it stopped.

The notebook was generated from the package source by `python/build_standalone_notebook.py`, so it matches the repository code; the theory behind each step is in `WTe2_TIFET_theory.ipynb`.

## Run settings

In [ ]:
BRANCH = "test_1"                 # GitHub branch to download the data from
RUN_STEPS = ["targets", "splits", "classical", "mlp", "unsupervised", "robustness",
             "uncertainty", "epoch_sensitivity"]           # remove a step to skip it
REUSE_GITHUB_RESULTS = []         # e.g. ["classical", "mlp", "epoch_sensitivity"]: download those results
                                  # from GitHub instead of recomputing (minutes instead of about an hour)
N_JOBS = 4                        # parallel fits for every step (Kaggle: 4 CPU cores, about 30 GB RAM)
REUSABLE = {'classical': ['metrics_classical.csv', 'predictions_classical.csv.gz'], 'mlp': ['metrics_mlp.csv', 'predictions_mlp.csv.gz', 'mlp_vs_classical.csv'], 'epoch_sensitivity': ['epoch_sensitivity.csv']}

## Setup: packages, working folder, data download from GitHub

In [ ]:
import json, os, subprocess, sys, time, urllib.request
from pathlib import Path
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyyaml", "psutil", "scikit-learn>=1.5"], check=True)
os.environ["PYTHONWARNINGS"] = "ignore"
import warnings; warnings.filterwarnings("ignore")

WORK = Path("/kaggle/working/wte2_run") if Path("/kaggle/working").exists() else Path.cwd() / "wte2_run"
RAW = "https://raw.githubusercontent.com/SKBiswas1998/ML-Project-for-ECE/{branch}/{path}"
def download(path: str) -> Path:
    """Fetch one file of the repository (data or saved results) into WORK, unless already there."""
    target = WORK / path
    if not target.exists():
        target.parent.mkdir(parents=True, exist_ok=True)
        urllib.request.urlretrieve(RAW.format(branch=BRANCH, path=path), target)
    return target
for name in ['ML_IV_train.csv', 'ML_IV_test.csv', 'ML_metrics_train.csv', 'ML_metrics_test.csv', 'NEGF_all.mat']:
    print("data:", download("ML_edge_vacancy_seed2026/" + name), flush=True)

## Configuration
Every constant of the pipeline (the repository's `python/config.yaml`), with the Kaggle parallelism applied below it. Edit here to change a setting.

In [ ]:
import yaml
CONFIG_YAML = r"""
# Single source of every constant used by the pipeline.
# Paths are relative to the repository root (the folder that holds python/).

paths:
  data_dir: ML_edge_vacancy_seed2026   # NEGF output, read-only
  results_dir: results                 # experiment CSVs (input to make_figures.py)
  figures_dir: figures
  cache_dir: results/cache             # intermediate fits, makes long runs resumable

data:
  iv_train: ML_IV_train.csv            # one row per gate point
  iv_test: ML_IV_test.csv
  metrics_train: ML_metrics_train.csv  # one row per curve (gate sweep)
  metrics_test: ML_metrics_test.csv
  negf_all: NEGF_all.mat               # layout info (vac_atoms, nested), scipy.io.loadmat
  curve_key: [T_K, vac_pct]            # identifies one gate sweep
  gate_column: VTG_minus_VBG_V         # lambda_eV and Ez_V_per_A are linear copies of it
  points_per_curve: 20                 # gate points per sweep, verified in Stage 0
  expected_rows:                       # verified in Stage 0; loader fails loudly if they change
    iv_train: 12760
    iv_test: 3200
    metrics_train: 638
    metrics_test: 160
  expected_seed: 2026                  # single random vacancy layout
  gate_grid_tolerance_V: 1.0e-9        # gate values of all curves must agree to this

physics:
  k_B_eV_per_K: 8.617333262e-5         # Boltzmann constant (CODATA 2018)
  V_DS_V: 0.05                         # drain bias (NEGF_all.mat: Vds); SPD = V_DS * Ioff
  dynamic_power: DED_over_tau          # P_dyn = DED_J / tau_d_s (the ONE convention)

targets:
  ioff_primary: last                   # 'last' = supplied Ioff (I at V = 1.9 V); 'min' = curve minimum
  ioff_secondary: min                  # reported alongside the primary
  swing:
    pchip_points: 2001                 # fine V grid for the PCHIP slope (~1 mV spacing over 1.9 V)
    min_drop_decades: 1.0              # branches falling less than this have no defined swing (NaN)
    check_T_K: [4, 40, 78]             # temperatures shown in the swing check figure
    check_vac_pct: [0.0, 0.6, 1.3, 2.0]  # clean, non-monotone (0.6/1.3), most damaged
    tangent_half_width_V: 0.15         # plotted tangent extends +/- this around the steepest point

splits:
  vac_step_pct: 0.1                    # spacing of the vacancy grid; levels are indexed round(vac/step)
  temp_train_max_K: 60                 # protocol 2 (forward-chaining): train T <= 60 K, test 62-78 K
  vac_block_holdout_pct: [0.8, 1.2]    # protocol 3a: middle block of consecutive levels held out
  vac_train_max_pct: 1.5               # protocol 3b: train <= 1.5 %, test 1.6-2.0 %
  holdout_embargo_levels: 1            # levels dropped from TRAINING next to the final vacancy test (Stage 7)
  cv_n_blocks: 5                       # hyperparameter CV: blocks of consecutive vacancy levels
  cv_embargo_levels: 1                 # levels dropped on each side of the held-out CV block
  temp_step_K: 2                       # spacing of the temperature grid; levels = round(T / step)
  cv_temp_n_blocks: 5                  # temperature_forward: CV blocks of consecutive temperatures (4-60 K)
  cv_temp_embargo_levels: 1            # temperatures dropped on each side of the held-out CV block

inputs:
  vacancy_features: [vac_pct]          # Stage 7: n_vac_atoms dropped; Stage 4 showed median MAE ratio 1.0 (results/encoding_comparison_stage4.csv)
  gate_decimals: 6                     # gate values rounded to this for exact per-gate grouping

experiments:
  point_target: log10_I                # per gate point, inputs (T, vacancy, gate)
  point_vacancy_features: [vac_pct]    # log10_I runs one encoding only: the per-gate GP costs ~170 s/job
  curve_targets: [log10_Ion_Ioff, log10_Ion_Ioff_min, log10_SPD, log10_DED, log10_tau_d,
                  log10_P_dyn, swing_mV_per_dec]   # per curve, inputs (T, vacancy)
  cv_scoring: neg_mean_squared_error
  mlp_targets: [log10_I, log10_Ion_Ioff, log10_Ion_Ioff_min, log10_SPD, log10_DED, log10_tau_d,
                log10_P_dyn, swing_mV_per_dec]   # Stage 7: every target, same as the classical models
  mlp_vacancy_feature: vac_pct         # Stage 4: encoding made no difference (median MAE ratio 1.0)
  mlp_iv_mode: curve_output           # log10_I: input (T, vac) -> 20 outputs per curve (638 rows, not 12,760)
  mlp_iv_max_epochs: 8000             # log10_I only: 1,500 was hit by every supplied-split seed; patience still decides

models:
  ridge:
    alpha: [1.0e-4, 1.0e-2, 1.0, 100.0]
  poly_ridge:
    degree: [2, 3, 4, 5]
    alpha: [1.0e-4, 1.0e-2, 1.0]
  boosting:                            # sklearn HistGradientBoostingRegressor
    max_depth: [2, 4, 6]
    max_iter: [100, 300]               # number of trees
    learning_rate: 0.1
  gp:                                  # one GP per gate point for log10_I; <= 638 points each
    kernels: [rbf, matern25]           # kernel family chosen by blocked CV (Stage 7); hyperparameters by marginal likelihood
    length_scale_bounds: [0.05, 100.0] # standardised units; 0.05 < one vacancy step (~0.17) -> levels may decorrelate
    noise_init: 1.0e-3                 # WhiteKernel start; absorbs non-smooth NEGF structure
    noise_bounds: [1.0e-8, 1.0]
    n_restarts_optimizer: 0            # each fit 3-6 s on 638 points; restarts did not change the optimum
  mlp:                                 # PyTorch, CPU, full-batch Adam on standardised X and y
    width: [64, 128]                   # units per hidden layer (tuned)
    depth: [2, 3]                      # hidden layers (tuned)
    weight_decay: [0.0, 1.0e-4]        # L2 penalty in AdamW (tuned)
    dropout: 0.0                       # fixed: data are noiseless simulation output
    learning_rate: 3.0e-3
    max_epochs: 1500
    patience: 100                      # early stopping: epochs without validation improvement
    eval_every: 5                      # epochs between validation checks (copying weights is costly)
    batch_size: 512                    # rows per Adam step; tasks with fewer rows train full-batch
    early_stop_fraction: 0.15          # share of training CURVES held out for early stopping
    seeds: [0, 1, 2, 3, 4]             # initialisation seeds; tuning uses the first
  seed: 0                              # random_state for every stochastic sklearn model

unsupervised:                          # descriptive only: fitted on all 798 curves, never used to predict
  pca_max_components: 10               # of 20 gate points
  cluster_variance_kept: 0.99          # cluster on the fewest PCs explaining this share of variance
  k_range: [2, 3, 4, 5, 6]
  gmm_covariance: full
  n_init: 10                           # restarts for k-means and the Gaussian mixture
  ion_ioff_class_edges: [4.0, 9.0]     # log10 Ion/Ioff: < 4 failed, 4-9 degraded, >= 9 working.
                                       # 9 = sparse region of the histogram (13 of 798 curves in 9-11);
                                       # 4 = our choice of a minimum usable on/off ratio (a convention)
  class_names: [failed, degraded, working]

robustness:                            # Gaussian noise on TEST inputs; model fitted once on clean data
  protocols: [supplied, vacancy_block] # interpolation and unseen-vacancy cases
  targets: [log10_Ion_Ioff]            # headline target
  models: [naive_vac_mean, naive_nn, ridge, poly_ridge, boosting, gp, mlp]
  noise_fractions: [0.0, 0.25, 0.5, 1.0]   # noise std as a fraction of one grid step per input
  repeats: 5                           # noise draws per level
  gate_step_V: 0.1                     # gate grid step (only used for per-gate-point targets)
  seed: 0

figures:
  headline_target: log10_Ion_Ioff      # Stage 7 decision; log10_Ion_Ioff_min reported as sensitivity
  error_map_models: [naive_vac_mean, boosting, gp, mlp]
  iv_sample_T_K:                       # a test temperature of each protocol (curves: lowest/middle/highest test vacancy)
    supplied: 40
    temperature_forward: 70
    vacancy_block: 40
    vacancy_extrapolation: 40

uncertainty:                           # Stage 8: cluster bootstrap over whole vacancy levels, saved predictions only
  n_boot: 2000                         # bootstrap samples
  confidence: 0.95                     # two-sided interval level (Bonferroni-adjusted one also reported)
  seed: 0

epoch_sensitivity:                     # Stage 8: do capped curve-level MLP jobs change with a higher cap?
  max_epochs: 8000                     # same limit as the I-V MLP; patience still decides when to stop
  jobs:                                # (protocol, target): most capped seeds in results/metrics_mlp.csv
    - [vacancy_extrapolation, log10_Ion_Ioff]
    - [vacancy_extrapolation, log10_Ion_Ioff_min]
    - [vacancy_extrapolation, swing_mV_per_dec]
    - [vacancy_block, swing_mV_per_dec]

compute:
  device: cpu                          # never assume CUDA
  torch_threads: 1                     # per MLP fit; fits run in parallel instead (small nets scale poorly)
  n_jobs: 4                            # parallel fits for the classical step (GP, grid search)
  mlp_n_jobs: 1                        # MLP fits run serially (4 parallel MLP fits ran the PC out of memory)
  num_workers: 0                       # DataLoader workers; 0 is required on Windows
"""
_CONFIG = yaml.safe_load(CONFIG_YAML)
_CONFIG["compute"]["n_jobs"] = N_JOBS        # notebook setting: parallel classical fits
_CONFIG["compute"]["mlp_n_jobs"] = N_JOBS    # and parallel MLP fits (enough memory on Kaggle)
REPO_ROOT = WORK

def load_config(path=None):
    """The configuration dict (a fresh copy each call, so callers cannot change it for others)."""
    return json.loads(json.dumps(_CONFIG))

def repo_path(cfg, key):
    """Absolute path of cfg['paths'][key] inside the working folder."""
    return REPO_ROOT / cfg["paths"][key]

## Imports used by all modules

In [ ]:
from __future__ import annotations

## Loading and validating the data
*Source: `python/wte2ml/data.py`*

Load and validate the NEGF dataset (CSVs + vacancy layout from NEGF_all.mat).

This is the only module that touches the raw files. Everything downstream gets
validated pandas tables or numpy arrays, so a changed or corrupted data file
fails here, loudly, instead of silently shifting a result.

Units follow the CSV column names: T_K in kelvin, vac_pct in percent of edge
atoms, VTG_minus_VBG_V in volt, I_A in ampere, log10_I = log10(I / 1 A).

In [ ]:
from dataclasses import dataclass
from itertools import pairwise
from typing import Any
import numpy as np
import pandas as pd
import scipy.io as sio
IV_COLUMNS = (
    "T_K", "vac_pct", "vac_pct_actual", "n_vac_atoms", "lambda_eV", "VTG_minus_VBG_V",
    "Ez_V_per_A", "I_A", "log10_I", "Cq_F_per_m", "Cq_F_per_m2", "CQ_F", "CT_F",
    "reliable", "seed",
)
METRIC_COLUMNS = (
    "T_K", "vac_pct", "vac_pct_actual", "n_vac_atoms", "Ion_A", "Ioff_A", "Ion_Ioff",
    "log10_Ion_Ioff", "CQ_avg_F", "C_eff_F", "tau_d_s", "DED_J", "SPD_W", "seed",
)

### `Dataset` (class)
The four supplied tables. ``split`` column added: 'train' or 'test'.

In [ ]:
@dataclass(frozen=True)
class Dataset:
    """The four supplied tables. ``split`` column added: 'train' or 'test'."""

    iv_train: pd.DataFrame
    iv_test: pd.DataFrame
    metrics_train: pd.DataFrame
    metrics_test: pd.DataFrame

    def iv_pooled(self) -> pd.DataFrame:
        """All 798 curves' gate points (train + test), for the pooled protocols."""
        return pd.concat([self.iv_train, self.iv_test], ignore_index=True)

    def metrics_pooled(self) -> pd.DataFrame:
        """All 798 curve-level rows (train + test)."""
        return pd.concat([self.metrics_train, self.metrics_test], ignore_index=True)

### `VacancyLayout` (class)
Vacancy configuration per level, from NEGF_all.mat.

In [ ]:
@dataclass(frozen=True)
class VacancyLayout:
    """Vacancy configuration per level, from NEGF_all.mat.

    vac_atoms[k] holds the removed atoms of level vac_pct[k] as 1-based MATLAB
    atom indices (kept as stored; never used to index a Python array).
    """

    vac_pct: np.ndarray            # (21,) nominal levels, percent
    n_vac: np.ndarray              # (21,) number of removed atoms
    vac_atoms: tuple[np.ndarray, ...]
    n_atoms: int                   # atoms in the device (1620)

    def is_nested(self) -> bool:
        """True if every level keeps all atoms removed at the previous level."""
        sets = [set(a.tolist()) for a in self.vac_atoms]
        return all(a <= b for a, b in pairwise(sets))

### `_read_table` (function)
Read one CSV, check the expected columns exist, tag rows with ``split``.

In [ ]:
def _read_table(path, columns: tuple[str, ...], split: str) -> pd.DataFrame:
    """Read one CSV, check the expected columns exist, tag rows with ``split``."""
    df = pd.read_csv(path)
    missing = set(columns) - set(df.columns)
    if missing:
        raise ValueError(f"{path.name}: missing columns {sorted(missing)}")
    return df.assign(split=split)

### `_check_table` (function)
Row count, no NaN, all points reliable, single layout seed.

In [ ]:
def _check_table(df: pd.DataFrame, name: str, n_expected: int, seed: int) -> None:
    """Row count, no NaN, all points reliable, single layout seed."""
    if len(df) != n_expected:
        raise ValueError(f"{name}: {len(df)} rows, expected {n_expected}")
    if df.isna().any().any():
        raise ValueError(f"{name}: contains NaN")
    if "reliable" in df and not (df["reliable"] == 1).all():
        raise ValueError(f"{name}: rows with reliable != 1")
    if not (df["seed"] == seed).all():
        raise ValueError(f"{name}: seed differs from {seed}")

### `check_curves` (function)
Every curve has ``points_per_curve`` points on one shared gate grid.

In [ ]:
def check_curves(iv: pd.DataFrame, cfg: dict[str, Any]) -> None:
    """Every curve has ``points_per_curve`` points on one shared gate grid."""
    key, gate = cfg["data"]["curve_key"], cfg["data"]["gate_column"]
    n_points = cfg["data"]["points_per_curve"]
    sizes = iv.groupby(key).size()
    if not (sizes == n_points).all():
        raise ValueError(f"curves with != {n_points} points: {sizes[sizes != n_points]}")
    grids = iv.sort_values(key + [gate]).groupby(key)[gate].apply(np.asarray)
    spread = np.ptp(np.stack(grids.to_numpy()), axis=0).max()
    if spread > cfg["data"]["gate_grid_tolerance_V"]:
        raise ValueError(f"gate grids differ between curves by {spread} V")

### `check_split_consistency` (function)
IV and metrics tables describe the same curves; train and test are disjoint.

In [ ]:
def check_split_consistency(ds: Dataset, cfg: dict[str, Any]) -> None:
    """IV and metrics tables describe the same curves; train and test are disjoint."""
    key = cfg["data"]["curve_key"]
    for split in ("train", "test"):
        iv_keys = set(map(tuple, getattr(ds, f"iv_{split}")[key].drop_duplicates().to_numpy()))
        m_keys = set(map(tuple, getattr(ds, f"metrics_{split}")[key].to_numpy()))
        if iv_keys != m_keys:
            raise ValueError(f"{split}: IV and metrics curve sets differ")
    overlap = set(map(tuple, ds.metrics_train[key].to_numpy())) & set(
        map(tuple, ds.metrics_test[key].to_numpy()))
    if overlap:
        raise ValueError(f"{len(overlap)} curves in both train and test")

### `load_dataset` (function)
Read the four CSVs and run every check. Returns validated tables.

In [ ]:
def load_dataset(cfg: dict[str, Any]) -> Dataset:
    """Read the four CSVs and run every check. Returns validated tables."""
    d, folder = cfg["data"], repo_path(cfg, "data_dir")
    tables = {}
    for name in ("iv_train", "iv_test", "metrics_train", "metrics_test"):
        cols = IV_COLUMNS if name.startswith("iv") else METRIC_COLUMNS
        df = _read_table(folder / d[name], cols, split=name.split("_")[1])
        _check_table(df, name, d["expected_rows"][name], d["expected_seed"])
        tables[name] = df
    ds = Dataset(**tables)
    check_curves(ds.iv_pooled(), cfg)
    check_split_consistency(ds, cfg)
    return ds

### `curve_matrix` (function)
Reshape gate-point rows into one row per curve.

In [ ]:
def curve_matrix(iv: pd.DataFrame, cfg: dict[str, Any], value: str = "log10_I"
                 ) -> tuple[pd.DataFrame, np.ndarray, np.ndarray]:
    """Reshape gate-point rows into one row per curve.

    Returns (keys, gate_V, Y): keys is a (n_curves, 2) table of T_K, vac_pct
    sorted by T then vacancy; gate_V the (20,) shared gate grid in volt; Y the
    (n_curves, 20) matrix of ``value``. Used by swing extraction and PCA.
    """
    key, gate = cfg["data"]["curve_key"], cfg["data"]["gate_column"]
    wide = iv.pivot_table(index=key, columns=gate, values=value).sort_index()
    return wide.index.to_frame(index=False), wide.columns.to_numpy(float), wide.to_numpy(float)

### `load_layout` (function)
Read vacancy layout arrays from NEGF_all.mat (cell arrays -> tuple of int arrays).

In [ ]:
def load_layout(cfg: dict[str, Any]) -> VacancyLayout:
    """Read vacancy layout arrays from NEGF_all.mat (cell arrays -> tuple of int arrays)."""
    raw = sio.loadmat(repo_path(cfg, "data_dir") / cfg["data"]["negf_all"], squeeze_me=True)
    atoms = tuple(np.atleast_1d(a).astype(int) for a in raw["vac_atoms"])
    return VacancyLayout(
        vac_pct=np.asarray(raw["vac_list"], float),
        n_vac=np.asarray(raw["n_vac"], int),
        vac_atoms=atoms,
        n_atoms=int(raw["n_at"]),
    )

## Learning targets and subthreshold swing
*Source: `python/wte2ml/targets.py`*

Learning targets derived from the validated tables.

Curve-level targets (one row per (T_K, vac_pct)) are all in log10, because
currents and powers span about 12 decades. Per-gate-point target log10_I is
used directly from the IV table.

Physics conventions (config ``physics``): SPD = V_DS * Ioff, DED = C_eff * V_DS^2,
tau_d = C_eff * V_DS / Ion, and dynamic power P_dyn = DED / tau_d.

In [ ]:
from typing import Any
import numpy as np
import pandas as pd
from scipy.interpolate import PchipInterpolator
CURVE_TARGETS = (
    "log10_Ion_Ioff", "log10_Ion_Ioff_min", "log10_SPD", "log10_DED",
    "log10_tau_d", "log10_P_dyn", "swing_mV_per_dec",
)

### `falling_branch` (function)
Indices of the first descent: from the on state (index 0) to the curve minimum.

In [ ]:
def falling_branch(log10_current: np.ndarray) -> slice:
    """Indices of the first descent: from the on state (index 0) to the curve minimum.

    The upturn after the minimum (present in 761 of 798 curves) is excluded,
    because it is a different mechanism from the switching we want to measure.
    """
    return slice(0, int(np.argmin(log10_current)) + 1)

### `steepest_swing` (function)
Subthreshold swing of one sweep, in mV/decade.

In [ ]:
def steepest_swing(gate_V: np.ndarray, log10_current: np.ndarray, n_fine: int,
                   min_drop: float) -> float:
    """Subthreshold swing of one sweep, in mV/decade.

    gate_V: (20,) increasing gate voltages in volt; log10_current: (20,) log10(I/A).
    PCHIP-interpolates log10 I(V) on the falling branch (shape-preserving: no
    overshoot between points) and returns min over V of |dV / dlog10 I| * 1000.
    Returns NaN when the branch falls less than ``min_drop`` decades (no switching).
    """
    br = falling_branch(log10_current)
    v, y = gate_V[br], log10_current[br]
    if v.size < 2 or y[0] - y[-1] < min_drop:
        return float("nan")
    fine = np.linspace(v[0], v[-1], n_fine)
    slope = PchipInterpolator(v, y).derivative()(fine)     # decades per volt
    return 1e3 / float(np.max(-slope))

### `raw_grid_swing` (function)
Swing from the steepest single grid step (mV/dec): an upper bound on the true value.

In [ ]:
def raw_grid_swing(gate_V: np.ndarray, log10_current: np.ndarray) -> float:
    """Swing from the steepest single grid step (mV/dec): an upper bound on the true value."""
    br = falling_branch(log10_current)
    step = -np.diff(log10_current[br]) / np.diff(gate_V[br])
    return 1e3 / float(step.max()) if step.size and step.max() > 0 else float("nan")

### `boltzmann_swing` (function)
Thermionic limit kT ln(10)/q in mV/dec (k_B in eV/K, so kT/q is in volt).

In [ ]:
def boltzmann_swing(T_K: np.ndarray | float, k_B_eV_per_K: float) -> np.ndarray:
    """Thermionic limit kT ln(10)/q in mV/dec (k_B in eV/K, so kT/q is in volt)."""
    return 1e3 * k_B_eV_per_K * np.asarray(T_K, float) * np.log(10)

### `swing_table` (function)
Per-curve swing (PCHIP and raw-grid) with the Boltzmann limit at that T.

In [ ]:
def swing_table(iv: pd.DataFrame, cfg: dict[str, Any]) -> pd.DataFrame:
    """Per-curve swing (PCHIP and raw-grid) with the Boltzmann limit at that T."""
    keys, gate_V, Y = curve_matrix(iv, cfg)
    sw = cfg["targets"]["swing"]
    pchip = [steepest_swing(gate_V, y, sw["pchip_points"], sw["min_drop_decades"]) for y in Y]
    raw = [raw_grid_swing(gate_V, y) for y in Y]
    return keys.assign(
        swing_mV_per_dec=pchip,
        swing_raw_mV_per_dec=raw,
        swing_boltzmann_mV_per_dec=boltzmann_swing(keys["T_K"], cfg["physics"]["k_B_eV_per_K"]),
        idx_min=np.argmin(Y, axis=1),
    )

### `curve_targets` (function)
One row per curve: inputs, split label and every log10 / swing target.

In [ ]:
def curve_targets(metrics: pd.DataFrame, iv: pd.DataFrame, cfg: dict[str, Any]) -> pd.DataFrame:
    """One row per curve: inputs, split label and every log10 / swing target.

    metrics, iv: validated tables covering the same curves (e.g. pooled).
    log10_Ion_Ioff uses the supplied Ioff (current at the last gate point);
    log10_Ion_Ioff_min uses the minimum current of the sweep.
    """
    key = cfg["data"]["curve_key"]
    i_min = iv.groupby(key)["I_A"].min().rename("Ioff_min_A")
    m = metrics.merge(i_min, left_on=key, right_index=True, validate="one_to_one")
    m = m.merge(swing_table(iv, cfg), on=key, validate="one_to_one")
    p_dyn = dynamic_power(m["DED_J"], m["tau_d_s"], cfg["physics"]["dynamic_power"])
    out = m[[*key, "vac_pct_actual", "n_vac_atoms", "split"]].assign(
        log10_Ion_Ioff=np.log10(m["Ion_A"] / m["Ioff_A"]),
        log10_Ion_Ioff_min=np.log10(m["Ion_A"] / m["Ioff_min_A"]),
        log10_SPD=np.log10(m["SPD_W"]),
        log10_DED=np.log10(m["DED_J"]),
        log10_tau_d=np.log10(m["tau_d_s"]),
        log10_P_dyn=np.log10(p_dyn),
        swing_mV_per_dec=m["swing_mV_per_dec"],
        swing_raw_mV_per_dec=m["swing_raw_mV_per_dec"],
        swing_boltzmann_mV_per_dec=m["swing_boltzmann_mV_per_dec"],
    )
    return out.sort_values(key, ignore_index=True)

### `dynamic_power` (function)
Dynamic power in watt. The single convention lives in config ``physics.dynamic_power``.

In [ ]:
def dynamic_power(ded_J: pd.Series, tau_s: pd.Series, convention: str) -> pd.Series:
    """Dynamic power in watt. The single convention lives in config ``physics.dynamic_power``."""
    if convention != "DED_over_tau":
        raise ValueError(f"unknown dynamic-power convention {convention!r}")
    return ded_J / tau_s

## Evaluation protocols and blocked cross-validation
*Source: `python/wte2ml/splits.py`*

Evaluation protocols and blocked cross-validation, all defined on CURVES.

A split is a pair of curve-key tables (T_K, vac_pct). Rows of any table (IV rows
or curve targets) are assigned by key, so the 20 gate points of one sweep can
never land on both sides. Random row splits are deliberately not offered.

Vacancy levels are compared through an integer level index round(vac_pct / step)
(step from config), which avoids float comparisons such as 0.30000000000000004.

In [ ]:
import hashlib
from dataclasses import dataclass
from typing import Any
import numpy as np
import pandas as pd
KEY = ["T_K", "vac_pct"]

### `vacancy_level` (function)
Integer vacancy-level index: with step 0.1, 0.0 % -> 0, 0.1 % -> 1, ..., 2.0 % -> 20.

In [ ]:
def vacancy_level(vac_pct: pd.Series | np.ndarray | list, step: float) -> np.ndarray:
    """Integer vacancy-level index: with step 0.1, 0.0 % -> 0, 0.1 % -> 1, ..., 2.0 % -> 20."""
    return np.rint(np.asarray(vac_pct, float) / step).astype(int)

### `curve_keys` (function)
Unique (T_K, vac_pct) of a table, sorted. Works for IV rows and curve rows.

In [ ]:
def curve_keys(table: pd.DataFrame) -> pd.DataFrame:
    """Unique (T_K, vac_pct) of a table, sorted. Works for IV rows and curve rows."""
    return table[KEY].drop_duplicates().sort_values(KEY, ignore_index=True)

### `_key_set` (function)
Set of (T_K, level index) pairs present in ``table``.

In [ ]:
def _key_set(table: pd.DataFrame, step: float) -> set[tuple[int, int]]:
    """Set of (T_K, level index) pairs present in ``table``."""
    return set(zip(table["T_K"].astype(int), vacancy_level(table["vac_pct"], step)))

### `Split` (class)
Named train/test partition of curve keys.

In [ ]:
@dataclass(frozen=True)
class Split:
    """Named train/test partition of curve keys.

    ``step`` is the vacancy grid spacing; ``cv_axis`` says along which input the
    hyperparameter CV inside ``train`` is blocked ("vacancy" or "temperature"), chosen
    to resemble the final test of the protocol.
    """

    name: str
    train: pd.DataFrame
    test: pd.DataFrame
    step: float
    cv_axis: str = "vacancy"

    def fingerprint(self) -> str:
        """Short hash of the exact train and test curves and CV axis (used in cache keys, so
        a cached result is reused only if it was computed on this identical split)."""
        parts = [self.name, self.cv_axis] + [sorted(_key_set(k, self.step)) for k in (self.train, self.test)]
        return hashlib.sha1(repr(parts).encode()).hexdigest()[:8]

    def apply(self, table: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
        """Rows of ``table`` whose curve is in train / in test (original row order kept)."""
        return self._rows(table, self.train), self._rows(table, self.test)

    def _rows(self, table: pd.DataFrame, keys: pd.DataFrame) -> pd.DataFrame:
        wanted = _key_set(keys, self.step)
        rows = zip(table["T_K"].astype(int), vacancy_level(table["vac_pct"], self.step))
        return table.loc[[k in wanted for k in rows]]

### `_make` (function)
Build a Split: ``test`` curves form the test set; ``drop`` curves are excluded from training only (the embargo).

In [ ]:
def _make(name: str, keys: pd.DataFrame, test: np.ndarray, drop: np.ndarray,
          step: float, cv_axis: str = "vacancy") -> Split:
    """Build a Split: ``test`` curves form the test set; ``drop`` curves are excluded
    from training only (the embargo)."""
    return Split(name, keys[~test & ~drop].reset_index(drop=True),
                 keys[test].reset_index(drop=True), step, cv_axis)

### `_near` (function)
Levels within ``margin`` grid steps of the closed range [lo, hi] (range included).

In [ ]:
def _near(levels: np.ndarray, lo: int, hi: int, margin: int) -> np.ndarray:
    """Levels within ``margin`` grid steps of the closed range [lo, hi] (range included)."""
    return (levels >= lo - margin) & (levels <= hi + margin)

### `supplied_split` (function)
Protocol 1: the supplied train/test labels (interpolation only).

In [ ]:
def supplied_split(curves: pd.DataFrame, cfg: dict[str, Any]) -> Split:
    """Protocol 1: the supplied train/test labels (interpolation only)."""
    step, keys = cfg["splits"]["vac_step_pct"], curve_keys(curves)
    test_set = _key_set(curves[curves["split"] == "test"], step)
    test = np.array([k in test_set for k in _key_set_list(keys, step)])
    return _make("supplied", keys, test, np.zeros(len(keys), bool), step)

### `_key_set_list` (function)
(T_K, level) pairs of ``keys`` in row order.

In [ ]:
def _key_set_list(keys: pd.DataFrame, step: float) -> list[tuple[int, int]]:
    """(T_K, level) pairs of ``keys`` in row order."""
    return list(zip(keys["T_K"].astype(int), vacancy_level(keys["vac_pct"], step)))

### `temperature_forward` (function)
Protocol 2 (forward-chaining in T): train T <= temp_train_max_K, test above it. Its hyperparameters are tuned by temperature-blocked CV (``cv_axis="temperature"``).

In [ ]:
def temperature_forward(curves: pd.DataFrame, cfg: dict[str, Any]) -> Split:
    """Protocol 2 (forward-chaining in T): train T <= temp_train_max_K, test above it.
    Its hyperparameters are tuned by temperature-blocked CV (``cv_axis="temperature"``)."""
    keys = curve_keys(curves)
    test = keys["T_K"].to_numpy() > cfg["splits"]["temp_train_max_K"]
    return _make("temperature_forward", keys, test, np.zeros(len(keys), bool),
                 cfg["splits"]["vac_step_pct"], cv_axis="temperature")

### `vacancy_block` (function)
Protocol 3a: hold out a block of consecutive vacancy levels at all temperatures.

In [ ]:
def vacancy_block(curves: pd.DataFrame, cfg: dict[str, Any]) -> Split:
    """Protocol 3a: hold out a block of consecutive vacancy levels at all temperatures."""
    s = cfg["splits"]
    keys = curve_keys(curves)
    lv = vacancy_level(keys["vac_pct"], s["vac_step_pct"])
    lo, hi = vacancy_level(s["vac_block_holdout_pct"], s["vac_step_pct"])
    test = _near(lv, lo, hi, 0)
    drop = _near(lv, lo, hi, s["holdout_embargo_levels"]) & ~test
    return _make("vacancy_block", keys, test, drop, s["vac_step_pct"])

### `vacancy_extrapolation` (function)
Protocol 3b: train on levels <= vac_train_max_pct, test on every higher level.

In [ ]:
def vacancy_extrapolation(curves: pd.DataFrame, cfg: dict[str, Any]) -> Split:
    """Protocol 3b: train on levels <= vac_train_max_pct, test on every higher level."""
    s = cfg["splits"]
    keys = curve_keys(curves)
    lv = vacancy_level(keys["vac_pct"], s["vac_step_pct"])
    cut = int(vacancy_level([s["vac_train_max_pct"]], s["vac_step_pct"])[0])
    test = lv > cut
    drop = ~test & (lv > cut - s["holdout_embargo_levels"])
    return _make("vacancy_extrapolation", keys, test, drop, s["vac_step_pct"])

### `all_protocols` (function)
The four final evaluation splits, in reporting order. ``curves`` = pooled 798.

In [ ]:
def all_protocols(curves: pd.DataFrame, cfg: dict[str, Any]) -> list[Split]:
    """The four final evaluation splits, in reporting order. ``curves`` = pooled 798."""
    return [supplied_split(curves, cfg), temperature_forward(curves, cfg),
            vacancy_block(curves, cfg), vacancy_extrapolation(curves, cfg)]

### `_cv_levels` (function)
Integer level of each curve along the CV axis, plus that axis' (n_blocks, embargo).

In [ ]:
def _cv_levels(keys: pd.DataFrame, cfg: dict[str, Any], axis: str) -> tuple[np.ndarray, int, int]:
    """Integer level of each curve along the CV axis, plus that axis' (n_blocks, embargo)."""
    s = cfg["splits"]
    if axis == "vacancy":
        return vacancy_level(keys["vac_pct"], s["vac_step_pct"]), s["cv_n_blocks"], s["cv_embargo_levels"]
    if axis == "temperature":
        levels = np.rint(keys["T_K"].to_numpy(float) / s["temp_step_K"]).astype(int)
        return levels, s["cv_temp_n_blocks"], s["cv_temp_embargo_levels"]
    raise ValueError(f"unknown cv axis {axis!r}")

### `blocked_cv_folds` (function)
Hyperparameter CV inside one training pool: blocks of consecutive levels of ``axis``.

In [ ]:
def blocked_cv_folds(train_keys: pd.DataFrame, cfg: dict[str, Any], axis: str = "vacancy"
                     ) -> list[Split]:
    """Hyperparameter CV inside one training pool: blocks of consecutive levels of ``axis``.

    The pool's distinct levels (vacancy levels, or temperatures) are cut into
    contiguous blocks. Fold i validates on block i and trains on the other levels minus
    those within the embargo (grid steps) of block i. For vacancy, nested layouts mean
    this REDUCES leakage between levels; it cannot remove it.
    """
    keys = curve_keys(train_keys)
    lv, n_blocks, embargo = _cv_levels(keys, cfg, axis)
    folds = []
    for i, block in enumerate(np.array_split(np.unique(lv), n_blocks)):
        val = np.isin(lv, block)
        drop = _near(lv, block.min(), block.max(), embargo) & ~val
        folds.append(_make(f"cv_fold{i}", keys, val, drop, cfg["splits"]["vac_step_pct"], axis))
    return folds

### `fold_indices` (function)
Positional (train_idx, val_idx) of ``table`` rows for each fold (GridSearchCV ``cv``).

In [ ]:
def fold_indices(table: pd.DataFrame, folds: list[Split]) -> list[tuple[np.ndarray, np.ndarray]]:
    """Positional (train_idx, val_idx) of ``table`` rows for each fold (GridSearchCV ``cv``).

    ``table`` must have a default RangeIndex (positions == labels); rows of curves that a
    fold drops (embargo) appear in neither array.
    """
    if not table.index.equals(pd.RangeIndex(len(table))):
        raise ValueError("fold_indices needs a table with a fresh RangeIndex")
    out = []
    for f in folds:
        tr, va = f.apply(table)
        out.append((tr.index.to_numpy(), va.index.to_numpy()))
    return out

## Metrics, skill and comparison tables
*Source: `python/wte2ml/evaluate.py`*

Error metrics. Targets are already log10 (or mV/dec for swing), so MAE/RMSE are
in decades (or mV/dec) directly.

In [ ]:
import numpy as np
import pandas as pd

### `regression_metrics` (function)
MAE, RMSE, MSE and R^2 of one prediction vector (any shape, flattened).

In [ ]:
def regression_metrics(y_true: np.ndarray, y_pred: np.ndarray) -> dict[str, float]:
    """MAE, RMSE, MSE and R^2 of one prediction vector (any shape, flattened)."""
    y, p = np.ravel(y_true).astype(float), np.ravel(y_pred).astype(float)
    err = p - y
    mse = float(np.mean(err**2))
    var = float(np.var(y))
    return {
        "MAE": float(np.mean(np.abs(err))),
        "RMSE": float(np.sqrt(mse)),
        "MSE": mse,
        "R2": 1.0 - mse / var if var > 0 else float("nan"),
        "n": int(y.size),
    }

### `skill_score` (function)
1 - MSE_model / MSE_best_naive: 0 = as good as the best naive baseline, 1 = perfect, negative = worse than the baseline.

In [ ]:
def skill_score(mse_model: float, mse_best_naive: float) -> float:
    """1 - MSE_model / MSE_best_naive: 0 = as good as the best naive baseline, 1 = perfect,
    negative = worse than the baseline."""
    return 1.0 - mse_model / mse_best_naive if mse_best_naive > 0 else float("nan")

### `comparison_table` (function)
One row per (protocol, target, model): MAE, RMSE, R2, skill (MLP = mean over seeds of the per-seed metrics, with MAE_std), the rank of the model by MAE within its task, and whether it is best.

In [ ]:
def comparison_table(classical: pd.DataFrame, mlp_per_seed: pd.DataFrame) -> pd.DataFrame:
    """One row per (protocol, target, model): MAE, RMSE, R2, skill (MLP = mean over seeds of the per-seed metrics,
    with MAE_std), the rank of the model by MAE within its task, and whether it is best.

    classical: results/metrics_classical.csv; mlp_per_seed: results/metrics_mlp.csv.
    """
    task = ["protocol", "feature", "target"]
    cols = ["MAE", "RMSE", "R2", "skill"]
    mlp = mlp_per_seed.groupby(task)[cols].mean().join(
        mlp_per_seed.groupby(task).MAE.std().rename("MAE_std")).reset_index().assign(model="mlp")
    both = pd.concat([classical[[*task, "model", *cols]], mlp], ignore_index=True)
    both["rank"] = both.groupby(task).MAE.rank(method="min").astype(int)
    naive = (classical[classical.model.isin(["naive_vac_mean", "naive_nn"])]
             .pivot_table(index=task, columns="model", values="MAE").add_prefix("MAE_").reset_index())
    both = both.merge(naive, on=task, how="left")   # both naive baselines shown on every row
    return both.assign(best=both["rank"] == 1).sort_values([*task, "rank"], ignore_index=True)

### `pooled_predictions` (function)
All test predictions in one table: classical models as stored, the MLP as the mean of its seeds (results/predictions_classical.csv.gz and predictions_mlp.csv.gz).

In [ ]:
def pooled_predictions(classical: pd.DataFrame, mlp: pd.DataFrame) -> pd.DataFrame:
    """All test predictions in one table: classical models as stored, the MLP as the mean of
    its seeds (results/predictions_classical.csv.gz and predictions_mlp.csv.gz)."""
    keys = ["protocol", "feature", "target", "T_K", "vac_pct", "gate_V"]
    m = (mlp.assign(gate_V=mlp.gate_V.fillna(-1)).groupby(keys, as_index=False)
         .agg(y_true=("y_true", "first"), y_pred=("y_pred", "mean")).assign(model="mlp", y_std=np.nan))
    m["gate_V"] = m.gate_V.replace(-1, np.nan)
    return pd.concat([classical, m], ignore_index=True)

## The MLP (PyTorch, CPU)
*Source: `python/wte2ml/neural.py`*

Small multi-layer perceptron in PyTorch with an sklearn-style interface (CPU only).

Exists so the MLP can be tuned by the same blocked-CV grid search and evaluated by
the same code as every classical model. Inputs and target are standardised inside
``fit`` (statistics from the training data only). Early stopping holds out whole
curves, never single gate points of a curve that is also used for training.

In [ ]:
import copy
import numpy as np
import torch
from sklearn.base import BaseEstimator, RegressorMixin
from torch import nn

### `build_network` (function)
depth x [Linear(width) -> SiLU -> Dropout], then Linear(n_out). SiLU is smooth, so predictions have no kinks from ReLU corners. n_out = 20 predicts a whole I-V curve.

In [ ]:
def build_network(n_in: int, n_out: int, width: int, depth: int, dropout: float) -> nn.Sequential:
    """depth x [Linear(width) -> SiLU -> Dropout], then Linear(n_out). SiLU is smooth, so
    predictions have no kinks from ReLU corners. n_out = 20 predicts a whole I-V curve."""
    layers: list[nn.Module] = []
    for i in range(depth):
        layers += [nn.Linear(n_in if i == 0 else width, width), nn.SiLU(), nn.Dropout(dropout)]
    return nn.Sequential(*layers, nn.Linear(width, n_out))

### `curve_holdout_mask` (function)
Boolean mask of rows whose curve (T_K, vacancy = X[:, :2]) is in a random ``fraction`` of curves. Used for early stopping only.

In [ ]:
def curve_holdout_mask(X: np.ndarray, fraction: float, rng: np.random.Generator) -> np.ndarray:
    """Boolean mask of rows whose curve (T_K, vacancy = X[:, :2]) is in a random
    ``fraction`` of curves. Used for early stopping only."""
    _, curve_id = np.unique(X[:, :2], axis=0, return_inverse=True)
    n_curves = curve_id.max() + 1
    held = rng.choice(n_curves, size=max(1, round(fraction * n_curves)), replace=False)
    return np.isin(curve_id.ravel(), held)

### `MLPRegressor` (class)
Fully connected regressor, mini-batch AdamW + MSE, early stopping on held-out curves.

In [ ]:
class MLPRegressor(RegressorMixin, BaseEstimator):
    """Fully connected regressor, mini-batch AdamW + MSE, early stopping on held-out curves."""

    def __init__(self, width: int = 64, depth: int = 2, dropout: float = 0.0,
                 weight_decay: float = 0.0, learning_rate: float = 3e-3, max_epochs: int = 1500,
                 patience: int = 100, eval_every: int = 5, batch_size: int = 512,
                 early_stop_fraction: float = 0.15, seed: int = 0, device: str = "cpu",
                 threads: int = 1) -> None:
        self.width, self.depth, self.dropout = width, depth, dropout
        self.weight_decay, self.learning_rate = weight_decay, learning_rate
        self.max_epochs, self.patience, self.eval_every = max_epochs, patience, eval_every
        self.batch_size, self.early_stop_fraction, self.seed = batch_size, early_stop_fraction, seed
        self.device, self.threads = device, threads

    def _tensor(self, a: np.ndarray) -> torch.Tensor:
        return torch.as_tensor(a, dtype=torch.float32, device=self.device)

    def fit(self, X: np.ndarray, y: np.ndarray) -> MLPRegressor:
        """X: (n, n_features) raw inputs; y: (n,) one target or (n, k) k targets per row
        (e.g. the 20 gate points of a curve). Returns self (best-epoch weights)."""
        torch.set_num_threads(self.threads)
        torch.manual_seed(self.seed)
        X, y = np.asarray(X, float), np.asarray(y, float)
        self.single_output_ = y.ndim == 1
        y = y.reshape(len(y), -1)
        self.x_mean_, self.x_std_ = X.mean(0), X.std(0) + 1e-12
        self.y_mean_, self.y_std_ = y.mean(0), y.std(0) + 1e-12   # per output column
        val = curve_holdout_mask(X, self.early_stop_fraction, np.random.default_rng(self.seed))
        Xs, ys = self._tensor((X - self.x_mean_) / self.x_std_), self._tensor((y - self.y_mean_) / self.y_std_)
        self.net_ = build_network(X.shape[1], y.shape[1], self.width, self.depth, self.dropout).to(self.device)
        self.best_epoch_, self.stop_epoch_, self.best_val_mse_ = self._train(
            Xs[~val], ys[~val], Xs[val], ys[val])
        # True if training ran out of epochs instead of being stopped by patience.
        self.hit_epoch_cap_ = self.stop_epoch_ == self.max_epochs - 1
        return self

    def _epoch(self, opt, loss_fn, Xt: torch.Tensor, yt: torch.Tensor, gen: torch.Generator) -> None:
        """One pass over the training rows in shuffled mini-batches (one batch if n <= batch_size)."""
        self.net_.train()
        for idx in torch.randperm(len(yt), generator=gen).split(self.batch_size):
            opt.zero_grad()
            loss_fn(self.net_(Xt[idx]), yt[idx]).backward()
            opt.step()

    def _train(self, Xt, yt, Xv, yv) -> tuple[int, int, float]:
        """Training loop; every ``eval_every`` epochs checks the held-out curves and keeps
        the best weights; stops after ``patience`` epochs without improvement.
        Returns (best epoch, last epoch run, best validation MSE)."""
        opt = torch.optim.AdamW(self.net_.parameters(), lr=self.learning_rate, weight_decay=self.weight_decay)
        gen = torch.Generator().manual_seed(self.seed)
        loss_fn, best, best_epoch, state = nn.MSELoss(), float("inf"), 0, None
        for epoch in range(self.max_epochs):
            self._epoch(opt, loss_fn, Xt, yt, gen)
            if epoch % self.eval_every:
                continue
            self.net_.eval()
            with torch.no_grad():
                v = loss_fn(self.net_(Xv), yv).item()
            if v < best:
                best, best_epoch, state = v, epoch, copy.deepcopy(self.net_.state_dict())
            elif epoch - best_epoch >= self.patience:
                break
        self.net_.load_state_dict(state)
        return best_epoch, epoch, best

    def predict(self, X: np.ndarray) -> np.ndarray:
        """(n,) or (n, k) predictions (same shape as the fitted y) in original units."""
        self.net_.eval()
        with torch.no_grad():
            z = self.net_(self._tensor((np.asarray(X, float) - self.x_mean_) / self.x_std_))
        out = z.cpu().numpy() * self.y_std_ + self.y_mean_
        return out[:, 0] if self.single_output_ else out

## All models behind one fit/predict interface
*Source: `python/wte2ml/models.py`*

Model zoo behind one interface: sklearn-style ``fit(X, y)`` / ``predict(X)``.

Feature matrix X columns (built by ``features``): [T_K, vacancy] for curve-level
targets, [T_K, vacancy, gate_V] for log10_I. Every model that needs it standardises
X inside its own Pipeline, so scaling statistics come from the training fold only.

Any object with fit/predict (and optionally predict(X, return_std=True)) plugs in,
which is the slot a later PINN would use (``Regressor`` protocol below).

In [ ]:
from typing import Any, Protocol
import numpy as np
import pandas as pd
from joblib import Parallel, delayed
from sklearn.base import BaseEstimator, RegressorMixin, clone
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, ConstantKernel, Matern, WhiteKernel
from sklearn.linear_model import Ridge
from sklearn.model_selection import GridSearchCV
from sklearn.neighbors import KNeighborsRegressor
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
NAIVE_MODELS = ("naive_vac_mean", "naive_nn")
MODELS = (*NAIVE_MODELS, "ridge", "poly_ridge", "boosting", "gp")

### `Regressor` (class)
What run_experiments needs from a model (sklearn estimators satisfy it).

In [ ]:
class Regressor(Protocol):
    """What run_experiments needs from a model (sklearn estimators satisfy it)."""

    def fit(self, X: np.ndarray, y: np.ndarray) -> Regressor: ...
    def predict(self, X: np.ndarray) -> np.ndarray: ...

### `features` (function)
X matrix (n_rows, 2 or 3): T_K [K], vacancy [% or atoms], optionally gate [V].

In [ ]:
def features(table: pd.DataFrame, vacancy_feature: str, gate_column: str | None) -> np.ndarray:
    """X matrix (n_rows, 2 or 3): T_K [K], vacancy [% or atoms], optionally gate [V]."""
    cols = ["T_K", vacancy_feature] + ([gate_column] if gate_column else [])
    return table[cols].to_numpy(float)

### `VacancyLevelMean` (class)
Naive baseline: mean target of the training curves at the nearest vacancy level.

In [ ]:
class VacancyLevelMean(RegressorMixin, BaseEstimator):
    """Naive baseline: mean target of the training curves at the nearest vacancy level.

    Uses only column 1 of X (vacancy). For a held-out level the nearest training
    level is used (ties -> the lower level), which is the honest naive guess.
    """

    def fit(self, X: np.ndarray, y: np.ndarray) -> VacancyLevelMean:
        s = pd.Series(np.asarray(y, float)).groupby(np.asarray(X)[:, 1]).mean()
        self.levels_, self.means_ = s.index.to_numpy(float), s.to_numpy(float)
        return self

    def predict(self, X: np.ndarray) -> np.ndarray:
        idx = np.argmin(np.abs(np.asarray(X)[:, [1]] - self.levels_[None, :]), axis=1)
        return self.means_[idx]

### `Gatewise` (class)
Fit one copy of ``estimator`` per gate voltage, on the (T, vacancy) columns.

In [ ]:
class Gatewise(RegressorMixin, BaseEstimator):
    """Fit one copy of ``estimator`` per gate voltage, on the (T, vacancy) columns.

    Turns any curve-level model into a log10_I model. Exists so the GP never sees
    more than one point per curve (<= 638 points, not 12,760) and so the naive
    baselines compare like with like at each gate point. Last X column = gate [V].
    """

    def __init__(self, estimator: Any = None, decimals: int = 6, n_jobs: int = 1) -> None:
        self.estimator = estimator
        self.decimals = decimals
        self.n_jobs = n_jobs

    def _gates(self, X: np.ndarray) -> np.ndarray:
        return np.round(np.asarray(X)[:, -1], self.decimals)

    def fit(self, X: np.ndarray, y: np.ndarray) -> Gatewise:
        g, y = self._gates(X), np.asarray(y, float)
        gates = np.unique(g)
        fitted = Parallel(n_jobs=self.n_jobs)(
            delayed(clone(self.estimator).fit)(X[g == v, :-1], y[g == v]) for v in gates)
        self.models_ = dict(zip(gates, fitted))
        return self

    def predict(self, X: np.ndarray, return_std: bool = False):
        g = self._gates(X)
        mean, std = np.full(len(g), np.nan), np.full(len(g), np.nan)
        for v in np.unique(g):
            m = g == v
            out = self.models_[v].predict(X[m, :-1], **({"return_std": True} if return_std else {}))
            mean[m], std[m] = (out if return_std else (out, np.nan))
        return (mean, std) if return_std else mean

### `_gp_kernel` (function)
Amplitude x (RBF | Matern 5/2) with one bounded length scale per input, plus white noise.

In [ ]:
def _gp_kernel(name: str, n_dims: int, cfg_gp: dict[str, Any]):
    """Amplitude x (RBF | Matern 5/2) with one bounded length scale per input, plus white noise."""
    ls, bounds = np.ones(n_dims), tuple(cfg_gp["length_scale_bounds"])
    base = {"rbf": RBF(ls, bounds), "matern25": Matern(ls, bounds, nu=2.5)}[name]
    return ConstantKernel(1.0) * base + WhiteKernel(cfg_gp["noise_init"], tuple(cfg_gp["noise_bounds"]))

### `_curve_model` (function)
Estimator and grid for a model acting on (T, vacancy) columns only.

In [ ]:
def _curve_model(name: str, cfg: dict[str, Any]) -> tuple[Any, dict[str, list]]:
    """Estimator and grid for a model acting on (T, vacancy) columns only."""
    m, seed = cfg["models"], cfg["models"]["seed"]
    if name == "naive_vac_mean":
        return VacancyLevelMean(), {}
    if name == "naive_nn":
        return make_pipeline(StandardScaler(), KNeighborsRegressor(n_neighbors=1)), {}
    if name == "gp":
        # Kernel family chosen by blocked CV (Stage 7 decision); length scales and noise of the
        # chosen kernel are still fitted by marginal likelihood inside each fit.
        gp = GaussianProcessRegressor(normalize_y=True, random_state=seed,
                                      n_restarts_optimizer=m["gp"]["n_restarts_optimizer"])
        kernels = [_gp_kernel(k, 2, m["gp"]) for k in m["gp"]["kernels"]]
        return make_pipeline(StandardScaler(), gp), {"gaussianprocessregressor__kernel": kernels}
    raise ValueError(name)

### `_joint_model` (function)
Estimator and grid for a model acting on all columns of X jointly.

In [ ]:
def _joint_model(name: str, cfg: dict[str, Any]) -> tuple[Any, dict[str, list]]:
    """Estimator and grid for a model acting on all columns of X jointly."""
    m, seed = cfg["models"], cfg["models"]["seed"]
    if name == "ridge":
        return make_pipeline(StandardScaler(), Ridge()), {"ridge__alpha": m["ridge"]["alpha"]}
    if name == "poly_ridge":
        pipe = Pipeline([("scale", StandardScaler()), ("poly", PolynomialFeatures()), ("ridge", Ridge())])
        return pipe, {"poly__degree": m["poly_ridge"]["degree"], "ridge__alpha": m["poly_ridge"]["alpha"]}
    if name == "boosting":
        b = m["boosting"]
        gbm = HistGradientBoostingRegressor(learning_rate=b["learning_rate"], random_state=seed)
        return gbm, {"max_depth": b["max_depth"], "max_iter": b["max_iter"]}
    raise ValueError(name)

### `_mlp_model` (function)
PyTorch MLP; width, depth and weight decay tuned by blocked CV. For log10_I the MLP step feeds it one row per curve with 20 outputs (see run_experiments).

In [ ]:
def _mlp_model(cfg: dict[str, Any]) -> tuple[Any, dict[str, list]]:
    """PyTorch MLP; width, depth and weight decay tuned by blocked CV. For log10_I the
    MLP step feeds it one row per curve with 20 outputs (see run_experiments)."""
    m, c = cfg["models"]["mlp"], cfg["compute"]
    est = MLPRegressor(dropout=m["dropout"], learning_rate=m["learning_rate"],
                       max_epochs=m["max_epochs"], patience=m["patience"],
                       eval_every=m["eval_every"], batch_size=m["batch_size"],
                       early_stop_fraction=m["early_stop_fraction"], seed=m["seeds"][0],
                       device=c["device"], threads=c["torch_threads"])
    return est, {"width": m["width"], "depth": m["depth"], "weight_decay": m["weight_decay"]}

### `build_model` (function)
(unfitted estimator, hyperparameter grid) for model ``name``.

In [ ]:
def build_model(name: str, cfg: dict[str, Any], pointwise: bool) -> tuple[Any, dict[str, list]]:
    """(unfitted estimator, hyperparameter grid) for model ``name``.

    pointwise=True means X has a gate column; curve-level models (naive, GP) are
    then wrapped in ``Gatewise`` and their grid keys get the ``estimator__`` prefix.
    """
    if name in ("ridge", "poly_ridge", "boosting"):
        return _joint_model(name, cfg)
    if name == "mlp":
        return _mlp_model(cfg)
    est, grid = _curve_model(name, cfg)
    if not pointwise:
        return est, grid
    wrapped = Gatewise(est, decimals=cfg["inputs"]["gate_decimals"], n_jobs=cfg["compute"]["n_jobs"])
    return wrapped, {f"estimator__{k}": v for k, v in grid.items()}

### `tune_and_fit` (function)
Pick hyperparameters by blocked CV (given fold indices), refit on all of X.

In [ ]:
def tune_and_fit(est: Any, grid: dict[str, list], X: np.ndarray, y: np.ndarray,
                 cv: list[tuple[np.ndarray, np.ndarray]], cfg: dict[str, Any]
                 ) -> tuple[Any, dict[str, Any], float]:
    """Pick hyperparameters by blocked CV (given fold indices), refit on all of X.

    Returns (fitted model, best params, best mean CV MSE). CV only exists to choose
    hyperparameters, so a model with no grid (naive baselines, GP) is fitted directly
    and its CV MSE is NaN.
    """
    if not grid:
        return clone(est).fit(X, y), {}, float("nan")
    search = GridSearchCV(est, grid or {}, cv=cv, scoring=cfg["experiments"]["cv_scoring"],
                          n_jobs=cfg["compute"]["n_jobs"], refit=True, error_score="raise")
    search.fit(X, y)
    return search.best_estimator_, search.best_params_, -float(search.best_score_)

## PCA and clustering
*Source: `python/wte2ml/unsupervised.py`*

PCA of the log10 I-V curves and clustering of devices (unsupervised learning).

Descriptive only: everything here is fitted on all 798 curves to see how much
structure the I-V curves have, and whether devices fall into natural groups that
match a physics label (on/off ratio classes). Nothing here feeds a prediction, so
it does not touch the train/test protocols.

Input is the (n_curves, 20) matrix of log10 I [decades] from ``data.curve_matrix``.
Columns share one unit, so PCA centres but does not rescale them.

In [ ]:
from typing import Any
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import adjusted_rand_score, silhouette_score
from sklearn.mixture import GaussianMixture

### `fit_pca` (function)
PCA of the curves (rows) over gate points (columns), mean-centred, not scaled.

In [ ]:
def fit_pca(Y: np.ndarray, n_components: int) -> PCA:
    """PCA of the curves (rows) over gate points (columns), mean-centred, not scaled."""
    return PCA(n_components=n_components).fit(Y)

### `explained_variance_table` (function)
Per component: share of total variance and cumulative share (both 0-1).

In [ ]:
def explained_variance_table(pca: PCA) -> pd.DataFrame:
    """Per component: share of total variance and cumulative share (both 0-1)."""
    ratio = pca.explained_variance_ratio_
    return pd.DataFrame({"component": np.arange(1, ratio.size + 1),
                         "explained_variance_ratio": ratio, "cumulative": np.cumsum(ratio)})

### `n_components_for` (function)
Fewest leading components whose cumulative explained variance reaches ``share`` (all fitted components if the share is never reached).

In [ ]:
def n_components_for(pca: PCA, share: float) -> int:
    """Fewest leading components whose cumulative explained variance reaches ``share``
    (all fitted components if the share is never reached)."""
    cumulative = np.cumsum(pca.explained_variance_ratio_)
    return int(min(np.searchsorted(cumulative, share) + 1, cumulative.size))

### `component_table` (function)
Long table of component loadings vs gate voltage, plus the mean curve (component 0). Shows what each component does to a curve, e.g. 'deepens the off state'.

In [ ]:
def component_table(pca: PCA, gate_V: np.ndarray) -> pd.DataFrame:
    """Long table of component loadings vs gate voltage, plus the mean curve (component 0).
    Shows what each component does to a curve, e.g. 'deepens the off state'."""
    rows = [pd.DataFrame({"component": 0, "gate_V": gate_V, "loading": pca.mean_})]
    rows += [pd.DataFrame({"component": i + 1, "gate_V": gate_V, "loading": v})
             for i, v in enumerate(pca.components_)]
    return pd.concat(rows, ignore_index=True)

### `ion_ioff_classes` (function)
Physics label per curve: 0 below edges[0], 1 between, 2 at or above edges[1].

In [ ]:
def ion_ioff_classes(log10_ratio: np.ndarray, edges: list[float]) -> np.ndarray:
    """Physics label per curve: 0 below edges[0], 1 between, 2 at or above edges[1]."""
    return np.digitize(np.asarray(log10_ratio, float), edges)

### `cluster` (function)
Cluster the rows of Z (PCA scores) by k-means or a Gaussian mixture. Returns (labels, BIC); BIC (lower = better fit per parameter) is NaN for k-means.

In [ ]:
def cluster(Z: np.ndarray, method: str, k: int, cfg_u: dict[str, Any], seed: int
            ) -> tuple[np.ndarray, float]:
    """Cluster the rows of Z (PCA scores) by k-means or a Gaussian mixture.
    Returns (labels, BIC); BIC (lower = better fit per parameter) is NaN for k-means."""
    if method == "kmeans":
        return KMeans(k, n_init=cfg_u["n_init"], random_state=seed).fit_predict(Z), float("nan")
    if method == "gmm":
        gmm = GaussianMixture(k, covariance_type=cfg_u["gmm_covariance"], n_init=cfg_u["n_init"],
                              random_state=seed).fit(Z)
        return gmm.predict(Z), float(gmm.bic(Z))
    raise ValueError(method)

### `cluster_scores` (function)
For each method and k: silhouette (-1..1, higher = better separated), BIC (GMM only) and adjusted Rand index against the Ion/Ioff classes (0 = chance, 1 = identical).

In [ ]:
def cluster_scores(Z: np.ndarray, classes: np.ndarray, cfg_u: dict[str, Any], seed: int
                   ) -> pd.DataFrame:
    """For each method and k: silhouette (-1..1, higher = better separated), BIC (GMM only)
    and adjusted Rand index against the Ion/Ioff classes (0 = chance, 1 = identical)."""
    rows = []
    for method in ("kmeans", "gmm"):
        for k in cfg_u["k_range"]:
            lab, bic = cluster(Z, method, k, cfg_u, seed)
            rows.append({"method": method, "k": k, "silhouette": float(silhouette_score(Z, lab)), "bic": bic,
                         "ari_vs_ion_ioff_classes": float(adjusted_rand_score(classes, lab)),
                         "smallest_cluster": int(np.bincount(lab).min())})
    return pd.DataFrame(rows)

## Input-noise robustness
*Source: `python/wte2ml/robustness.py`*

Input-robustness test: how much does a fitted model's error grow when its test inputs
are perturbed by Gaussian noise?

Motivation (project proposal): a surrogate is only useful if a small error in the inputs
(a temperature reading off by a kelvin, a vacancy density known only roughly) does not
change its prediction wildly. Noise is added to the TEST inputs only; the model is fitted
once on clean training data. Noise is measured in grid steps of each input, so one
number means the same thing for temperature (2 K steps) and vacancy (0.1 % steps).

In [ ]:
from typing import Any
import numpy as np
import pandas as pd

### `input_steps` (function)
Grid step of each X column, in its own unit: [T step K, vacancy step %][, gate step V].

In [ ]:
def input_steps(cfg: dict[str, Any], n_columns: int) -> np.ndarray:
    """Grid step of each X column, in its own unit: [T step K, vacancy step %][, gate step V]."""
    s = cfg["splits"]
    steps = [s["temp_step_K"], s["vac_step_pct"], cfg["robustness"]["gate_step_V"]]
    return np.asarray(steps[:n_columns], float)

### `noisy` (function)
X + independent Gaussian noise with standard deviation sigma[j] in column j.

In [ ]:
def noisy(X: np.ndarray, sigma: np.ndarray, rng: np.random.Generator) -> np.ndarray:
    """X + independent Gaussian noise with standard deviation sigma[j] in column j."""
    return X + rng.standard_normal(X.shape) * sigma[None, :]

### `noise_curve` (function)
Test MAE for each noise level (fraction of a grid step) and repeat.

In [ ]:
def noise_curve(model: Any, X_test: np.ndarray, y_test: np.ndarray, steps: np.ndarray,
                fractions: list[float], repeats: int, seed: int) -> pd.DataFrame:
    """Test MAE for each noise level (fraction of a grid step) and repeat.

    Fraction 0 is evaluated once (it is the clean test error). Returns one row per
    (noise_fraction, repeat) with MAE and RMSE in the target's units.
    """
    rng = np.random.default_rng(seed)
    rows = []
    for frac in fractions:
        for rep in range(1 if frac == 0 else repeats):
            X = X_test if frac == 0 else noisy(X_test, frac * steps, rng)
            m = regression_metrics(y_test, np.ravel(model.predict(X)))
            rows.append({"noise_fraction": frac, "repeat": rep, "MAE": m["MAE"], "RMSE": m["RMSE"]})
    return pd.DataFrame(rows)

### `summarise` (function)
Mean and std of MAE over repeats per noise level, plus the increase over clean MAE.

In [ ]:
def summarise(curves: pd.DataFrame, keys: list[str]) -> pd.DataFrame:
    """Mean and std of MAE over repeats per noise level, plus the increase over clean MAE."""
    s = curves.groupby([*keys, "noise_fraction"]).MAE.agg(MAE_mean="mean", MAE_std="std").reset_index()
    clean = s[s.noise_fraction == 0].set_index(keys).MAE_mean.rename("MAE_clean")
    s = s.merge(clean, left_on=keys, right_index=True)
    return s.assign(MAE_increase=s.MAE_mean - s.MAE_clean)

## Cluster bootstrap and GP coverage
*Source: `python/wte2ml/uncertainty.py`*

Uncertainty of the reported test errors, computed from saved predictions (no refits).

Cluster bootstrap: curves of one vacancy level share one layout and are strongly
dependent, so the resampling unit is the whole vacancy LEVEL, never a single curve
(resampling curves would treat 38 near-copies as 38 independent samples and give
falsely tight intervals). With few levels in a test set (5 for the vacancy tests) the
intervals are coarse and wide; that is the honest answer, not a defect.

GP coverage: share of test points whose truth lies inside the GP's nominal 68 % and
95 % predictive intervals (mean +/- 1 and 1.96 std).

In [ ]:
from typing import Any
import numpy as np
import pandas as pd
Z68, Z95 = 1.0, 1.959964   # standard-normal half-widths of the 68.27 % and 95 % intervals

### `level_sums` (function)
Per vacancy level: sum of |error| and number of test rows (the bootstrap clusters).

In [ ]:
def level_sums(pred: pd.DataFrame, step: float) -> pd.DataFrame:
    """Per vacancy level: sum of |error| and number of test rows (the bootstrap clusters)."""
    level = np.rint(pred.vac_pct.to_numpy(float) / step).astype(int)
    err = (pred.y_pred - pred.y_true).abs().to_numpy()
    return pd.DataFrame({"level": level, "abs_err": err}).groupby("level").abs_err.agg(["sum", "count"])

### `resample_levels` (function)
(n_boot, n_levels) indices: each bootstrap sample draws n_levels levels with replacement.

In [ ]:
def resample_levels(n_levels: int, n_boot: int, rng: np.random.Generator) -> np.ndarray:
    """(n_boot, n_levels) indices: each bootstrap sample draws n_levels levels with replacement."""
    return rng.integers(0, n_levels, size=(n_boot, n_levels))

### `bootstrap_mae` (function)
MAE of each bootstrap sample: pooled |error| over the drawn levels / their row count.

In [ ]:
def bootstrap_mae(sums: pd.DataFrame, idx: np.ndarray) -> np.ndarray:
    """MAE of each bootstrap sample: pooled |error| over the drawn levels / their row count."""
    s, c = sums["sum"].to_numpy(), sums["count"].to_numpy()
    return s[idx].sum(axis=1) / c[idx].sum(axis=1)

### `percentile_ci` (function)
Two-sided percentile interval at confidence ``level`` (e.g. 0.95).

In [ ]:
def percentile_ci(samples: np.ndarray, level: float) -> tuple[float, float]:
    """Two-sided percentile interval at confidence ``level`` (e.g. 0.95)."""
    lo, hi = np.percentile(samples, [50 * (1 - level), 50 * (1 + level)])
    return float(lo), float(hi)

### `gp_coverage` (function)
Empirical coverage of the nominal 68 % and 95 % GP intervals, plus the mean predictive std. pred: rows with y_true, y_pred, y_std (GP only).

In [ ]:
def gp_coverage(pred: pd.DataFrame) -> dict[str, float]:
    """Empirical coverage of the nominal 68 % and 95 % GP intervals, plus the mean predictive std.
    pred: rows with y_true, y_pred, y_std (GP only)."""
    z = ((pred.y_true - pred.y_pred).abs() / pred.y_std).to_numpy()
    return {"coverage_68": float(np.mean(z <= Z68)), "coverage_95": float(np.mean(z <= Z95)),
            "mean_std": float(pred.y_std.mean()), "n": len(z)}

### `task_bootstrap` (function)
For one task (protocol, target): MAE with cluster-bootstrap CI for each model, and the paired difference MAE_model - MAE_reference on the SAME resampled levels, with a 95 % and a Bonferroni-adjusted interval (n_comparisons = number of model-vs-reference tests reported).

In [ ]:
def task_bootstrap(pred: pd.DataFrame, models: list[str], reference: str, cfg_u: dict[str, Any],
                   step: float, n_comparisons: int) -> pd.DataFrame:
    """For one task (protocol, target): MAE with cluster-bootstrap CI for each model, and the
    paired difference MAE_model - MAE_reference on the SAME resampled levels, with a 95 % and a
    Bonferroni-adjusted interval (n_comparisons = number of model-vs-reference tests reported)."""
    sums = {m: level_sums(pred[pred.model == m], step) for m in models}
    levels = sums[reference].index
    idx = resample_levels(len(levels), cfg_u["n_boot"], np.random.default_rng(cfg_u["seed"]))
    boot = {m: bootstrap_mae(sums[m].reindex(levels), idx) for m in models}
    level_bonf = 1 - (1 - cfg_u["confidence"]) / n_comparisons
    rows = []
    for m in models:
        diff = boot[m] - boot[reference]
        rows.append({"model": m, "n_levels": len(levels),
                     "MAE": float(sums[m]["sum"].sum() / sums[m]["count"].sum()),
                     **dict(zip(("MAE_lo", "MAE_hi"), percentile_ci(boot[m], cfg_u["confidence"]))),
                     **dict(zip(("diff_lo", "diff_hi"), percentile_ci(diff, cfg_u["confidence"]))),
                     **dict(zip(("diff_lo_bonf", "diff_hi_bonf"), percentile_ci(diff, level_bonf)))})
    return pd.DataFrame(rows).assign(reference=reference, n_comparisons=n_comparisons)

## Ridge regression from scratch
*Source: `python/wte2ml/ridge_numpy.py`*

Ridge regression written out in NumPy, to show what the sklearn model in the pipeline computes.

Same model as models.build_model("ridge"): StandardScaler (training mean, population std)
followed by sklearn Ridge with an unpenalised intercept. In standardised inputs
Z = (X - mean) / std and centred target, the coefficients solve

    (Z^T Z + alpha I) beta = Z^T (y - mean(y)),   intercept = mean(y)  (on the Z scale).

A test checks that predictions and coefficients agree with sklearn to numerical precision.

In [ ]:
from dataclasses import dataclass
import numpy as np

### `RidgeFit` (class)
Fitted ridge model. beta: (p,) coefficients on standardised inputs; intercept: mean of y.

In [ ]:
@dataclass(frozen=True)
class RidgeFit:
    """Fitted ridge model. beta: (p,) coefficients on standardised inputs; intercept: mean of y."""

    x_mean: np.ndarray
    x_std: np.ndarray
    beta: np.ndarray
    intercept: float
    alpha: float

    def predict(self, X: np.ndarray) -> np.ndarray:
        """(n,) predictions for raw inputs X (n, p)."""
        return ((np.asarray(X, float) - self.x_mean) / self.x_std) @ self.beta + self.intercept

### `_standardise` (function)
Z, mean, std with sklearn StandardScaler's convention (ddof = 0; constant columns -> std 1).

In [ ]:
def _standardise(X: np.ndarray) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    """Z, mean, std with sklearn StandardScaler's convention (ddof = 0; constant columns -> std 1)."""
    mean, std = X.mean(axis=0), X.std(axis=0)
    std = np.where(std == 0, 1.0, std)
    return (X - mean) / std, mean, std

### `fit_ridge` (function)
Closed-form ridge on standardised X; X (n, p) raw inputs, y (n,) target, alpha >= 0.

In [ ]:
def fit_ridge(X: np.ndarray, y: np.ndarray, alpha: float) -> RidgeFit:
    """Closed-form ridge on standardised X; X (n, p) raw inputs, y (n,) target, alpha >= 0."""
    X, y = np.asarray(X, float), np.asarray(y, float)
    Z, mean, std = _standardise(X)
    yc = y - y.mean()
    beta = np.linalg.solve(Z.T @ Z + alpha * np.eye(Z.shape[1]), Z.T @ yc)
    return RidgeFit(mean, std, beta, float(y.mean()), float(alpha))

### `effective_dof` (function)
Effective number of parameters, trace of the hat matrix: sum_j d_j^2 / (d_j^2 + alpha), with d_j the singular values of standardised X. Equals p at alpha = 0, falls to 0 as alpha grows.

In [ ]:
def effective_dof(X: np.ndarray, alpha: float) -> float:
    """Effective number of parameters, trace of the hat matrix: sum_j d_j^2 / (d_j^2 + alpha),
    with d_j the singular values of standardised X. Equals p at alpha = 0, falls to 0 as alpha grows."""
    d = np.linalg.svd(_standardise(np.asarray(X, float))[0], compute_uv=False)
    return float(np.sum(d**2 / (d**2 + alpha)))

### `coefficient_path` (function)
(len(alphas), p) coefficients for each alpha: how the penalty shrinks them towards 0.

In [ ]:
def coefficient_path(X: np.ndarray, y: np.ndarray, alphas: np.ndarray) -> np.ndarray:
    """(len(alphas), p) coefficients for each alpha: how the penalty shrinks them towards 0."""
    return np.vstack([fit_ridge(X, y, a).beta for a in alphas])

## Plotting primitives
*Source: `python/wte2ml/plots.py`*

Plotting functions for the Stage 7 report figures.

Each function draws on a given matplotlib Axes from tables that came from
results/*.csv (passed in as DataFrames), so the figures can never compute their own
numbers: make_figures.py only reads the CSVs, calls these, and saves the PNGs.

In [ ]:
import numpy as np
import pandas as pd
KEY = ["T_K", "vac_pct"]

### `heatmap` (function)
Colour map of ``value`` over the T x vacancy grid (T on y, vacancy on x). Curves absent from ``table`` (e.g. not in a test set) are left blank.

In [ ]:
def heatmap(ax, table: pd.DataFrame, value: str, title: str, cmap: str = "viridis",
            vmin: float | None = None, vmax: float | None = None):
    """Colour map of ``value`` over the T x vacancy grid (T on y, vacancy on x).
    Curves absent from ``table`` (e.g. not in a test set) are left blank."""
    grid = table.assign(vac=table.vac_pct.round(1)).pivot_table(index="T_K", columns="vac", values=value)
    im = ax.imshow(grid.to_numpy(), origin="lower", aspect="auto", cmap=cmap, vmin=vmin, vmax=vmax,
                   extent=[grid.columns.min() - 0.05, grid.columns.max() + 0.05,
                           grid.index.min() - 1, grid.index.max() + 1])
    ax.set(xlabel="edge vacancy (%)", ylabel="T (K)", title=title)
    return im

### `predicted_vs_true` (function)
Scatter of predicted vs true with the y = x line; MAE in the corner.

In [ ]:
def predicted_vs_true(ax, pred: pd.DataFrame, title: str, lims: tuple[float, float]) -> None:
    """Scatter of predicted vs true with the y = x line; MAE in the corner."""
    ax.scatter(pred.y_true, pred.y_pred, s=5, alpha=0.6)
    ax.plot(lims, lims, color="0.4", lw=0.8)
    mae = float(np.mean(np.abs(pred.y_pred - pred.y_true)))
    ax.text(0.04, 0.92, f"MAE {mae:.3f}", transform=ax.transAxes, fontsize=7)
    ax.set(xlim=lims, ylim=lims, title=title)
    ax.title.set_fontsize(8)
    ax.tick_params(labelsize=6)

### `error_profile` (function)
Mean absolute error of each model grouped by ``by`` (T_K or vac_pct), one line per model.

In [ ]:
def error_profile(ax, pred: pd.DataFrame, by: str, models: list[str], title: str) -> None:
    """Mean absolute error of each model grouped by ``by`` (T_K or vac_pct), one line per model."""
    for model in models:
        sub = pred[pred.model == model]
        g = sub.assign(err=(sub.y_pred - sub.y_true).abs(), key=sub[by].round(1)).groupby("key").err.mean()
        ax.plot(g.index, g.to_numpy(), "o-", ms=3, lw=1, label=model)
    ax.set(xlabel={"T_K": "T (K)", "vac_pct": "edge vacancy (%)"}[by], ylabel="MAE (decades)", title=title)
    ax.title.set_fontsize(8)

### `iv_with_bands` (function)
One I-V curve: NEGF truth, the GP mean with a +/- 2 std band, other models' predictions.

In [ ]:
def iv_with_bands(ax, curve: pd.DataFrame, models: list[str], title: str) -> None:
    """One I-V curve: NEGF truth, the GP mean with a +/- 2 std band, other models' predictions."""
    truth = curve[curve.model == models[0]].sort_values("gate_V")
    ax.plot(truth.gate_V, truth.y_true, "ko", ms=3, label="NEGF")
    for model in models:
        sub = curve[curve.model == model].sort_values("gate_V")
        line, = ax.plot(sub.gate_V, sub.y_pred, "-", lw=1.2, label=model)
        if model == "gp":
            ax.fill_between(sub.gate_V, sub.y_pred - 2 * sub.y_std, sub.y_pred + 2 * sub.y_std,
                            color=line.get_color(), alpha=0.25, label="GP +/- 2 std")
    ax.set(xlabel="V_TG - V_BG (V)", ylabel="log10 I (A)", title=title)
    ax.title.set_fontsize(8)

### `robustness_lines` (function)
Test MAE vs input-noise level (fraction of one grid step), mean +/- std over repeats.

In [ ]:
def robustness_lines(ax, summary: pd.DataFrame, title: str) -> None:
    """Test MAE vs input-noise level (fraction of one grid step), mean +/- std over repeats."""
    for model, sub in summary.groupby("model"):
        sub = sub.sort_values("noise_fraction")
        ax.errorbar(sub.noise_fraction, sub.MAE_mean, yerr=sub.MAE_std.fillna(0), marker="o",
                    ms=3, lw=1, capsize=2, label=model)
    ax.set(xlabel="input noise (std, fraction of one grid step)", ylabel="test MAE (decades)", title=title)
    ax.title.set_fontsize(8)

### `coefficient_path_lines` (function)
Ridge coefficients (standardised inputs) vs the penalty alpha, one line per feature.

In [ ]:
def coefficient_path_lines(ax, alphas: np.ndarray, path: np.ndarray, names: list[str]) -> None:
    """Ridge coefficients (standardised inputs) vs the penalty alpha, one line per feature."""
    for j, name in enumerate(names):
        ax.plot(alphas, path[:, j], lw=1, label=name)
    ax.set_xscale("log")
    ax.axhline(0, color="0.6", lw=0.8)
    ax.set(xlabel="penalty alpha", ylabel="coefficient (standardised input)", title="Ridge coefficient path")
    ax.legend(fontsize=6, ncol=2)

### `coverage_bars` (function)
Empirical coverage of the GP's 68 % and 95 % intervals per protocol, against nominal lines.

In [ ]:
def coverage_bars(ax, cover: pd.DataFrame, target: str) -> None:
    """Empirical coverage of the GP's 68 % and 95 % intervals per protocol, against nominal lines."""
    sub = cover[cover.target == target].reset_index(drop=True)
    x = np.arange(len(sub))
    ax.bar(x - 0.2, sub.coverage_68, 0.4, label="inside +/- 1 std")
    ax.bar(x + 0.2, sub.coverage_95, 0.4, label="inside +/- 1.96 std")
    for level, style in ((0.6827, "--"), (0.95, ":")):
        ax.axhline(level, color="0.3", ls=style, lw=1)
    ax.set_xticks(x, sub.protocol, rotation=15, fontsize=7)
    ax.set(ylim=(0, 1.05), ylabel="share of test points", title=f"GP interval coverage: {target}")
    ax.legend(fontsize=7)

## Experiment steps (each writes results/*.csv)
*Source: `python/run_experiments.py`*

Run pipeline steps and write results/*.csv. From the repo root:

    .\.venv\Scripts\python python\run_experiments.py targets

Each step writes its own CSVs and times itself. Later stages add model steps.

In [ ]:
import argparse
import hashlib
import json
import os
import time
from collections.abc import Callable
from functools import partial
from pathlib import Path
from typing import Any
import numpy as np
import pandas as pd
import psutil
from joblib import Parallel, delayed
from sklearn.base import clone

### `step_targets` (function)
Curve-level targets for all 798 curves plus a per-vacancy swing summary.

In [ ]:
def step_targets(cfg: dict[str, Any], out_dir: Path) -> list[Path]:
    """Curve-level targets for all 798 curves plus a per-vacancy swing summary."""
    ds = load_dataset(cfg)
    table = curve_targets(ds.metrics_pooled(), ds.iv_pooled(), cfg)
    summary = table.groupby("vac_pct").agg(
        swing_median_mV_per_dec=("swing_mV_per_dec", "median"),
        swing_raw_median_mV_per_dec=("swing_raw_mV_per_dec", "median"),
        log10_Ion_Ioff_median=("log10_Ion_Ioff", "median"),
        log10_Ion_Ioff_min_median=("log10_Ion_Ioff_min", "median"),
    ).reset_index()
    paths = [out_dir / "curve_targets.csv", out_dir / "swing_by_vacancy.csv"]
    table.to_csv(paths[0], index=False)
    summary.to_csv(paths[1], index=False)
    return paths

### `_describe` (function)
One summary row: sizes and T / vacancy ranges of both sides of a split.

In [ ]:
def _describe(split: Split, protocol: str) -> dict[str, Any]:
    """One summary row: sizes and T / vacancy ranges of both sides of a split."""
    row: dict[str, Any] = {"protocol": protocol, "split": split.name}
    for side in ("train", "test"):
        k = getattr(split, side)
        levels = sorted(k["vac_pct"].round(1).unique())
        row |= {f"{side}_curves": len(k), f"{side}_T_K": f"{k.T_K.min()}-{k.T_K.max()}",
                f"{side}_vac_pct": " ".join(f"{v:g}" for v in levels)}
    return row

### `step_splits` (function)
Every final split and every CV fold inside its training pool, as one table.

In [ ]:
def step_splits(cfg: dict[str, Any], out_dir: Path) -> list[Path]:
    """Every final split and every CV fold inside its training pool, as one table."""
    ds = load_dataset(cfg)
    curves = curve_targets(ds.metrics_pooled(), ds.iv_pooled(), cfg)
    rows = []
    for split in all_protocols(curves, cfg):
        rows.append(_describe(split, split.name))
        rows += [_describe(f, split.name) for f in blocked_cv_folds(split.train, cfg, split.cv_axis)]
    path = out_dir / "split_summary.csv"
    pd.DataFrame(rows).to_csv(path, index=False)
    return [path]

In [ ]:
GZIP = {"method": "gzip", "mtime": 0}   # fixed timestamp: identical results give identical files in git
# config keys only the MLP step uses
MLP_KEYS = ("mlp", "mlp_targets", "mlp_vacancy_feature", "mlp_iv_mode", "mlp_iv_max_epochs")
# keys that change only the log10_I MLP jobs: left out of the MLP hash, put in those jobs' file names
IV_ONLY_KEYS = ("mlp_iv_max_epochs",)

### `_config_hash` (function)
Short hash of the config settings that change ``step``'s results (cache key).

In [ ]:
def _config_hash(cfg: dict[str, Any], step: str) -> str:
    """Short hash of the config settings that change ``step``'s results (cache key).

    The classical step ignores MLP-only keys (and vice versa for model settings), so
    adding or tuning the MLP does not invalidate the 19-minute classical cache.
    """
    parts = {k: cfg[k] for k in ("data", "targets", "splits", "inputs", "experiments", "models")}
    if step == "mlp":
        parts["experiments"] = {k: v for k, v in parts["experiments"].items() if k not in IV_ONLY_KEYS}
    if step == "classical":
        parts = {k: ({kk: vv for kk, vv in v.items() if kk not in MLP_KEYS} if isinstance(v, dict) else v)
                 for k, v in parts.items()}
    return hashlib.sha1(json.dumps(parts, sort_keys=True).encode()).hexdigest()[:8]

### `_predict` (function)
Mean prediction, plus predictive std for the GP (NaN for other models).

In [ ]:
def _predict(model: Any, X: np.ndarray, name: str) -> tuple[np.ndarray, np.ndarray]:
    """Mean prediction, plus predictive std for the GP (NaN for other models)."""
    if name == "gp":
        return model.predict(X, return_std=True)
    return model.predict(X), np.full(len(X), np.nan)

### `_tuned` (function)
Build ``model_name``, tune it by blocked CV on ``d`` (from _task_data/_mlp_task_data), refit on the whole training pool. Shared by the classical, MLP and robustness steps. The MLP gets the log10_I-only epoch limit; per-gate models get the gate column.

In [ ]:
def _tuned(cfg: dict[str, Any], model_name: str, target: str, d: dict[str, Any]
           ) -> tuple[Any, dict[str, Any], float]:
    """Build ``model_name``, tune it by blocked CV on ``d`` (from _task_data/_mlp_task_data),
    refit on the whole training pool. Shared by the classical, MLP and robustness steps.
    The MLP gets the log10_I-only epoch limit; per-gate models get the gate column."""
    est, grid = build_model(model_name, cfg, pointwise=d["gate"] is not None)
    if model_name == "mlp" and target == cfg["experiments"]["point_target"]:
        est.set_params(max_epochs=cfg["experiments"]["mlp_iv_max_epochs"])
    return tune_and_fit(est, grid, d["X_train"], d["y_train"], d["cv"], cfg)

### `run_job` (function)
Tune one model by blocked CV inside ``split.train``, refit, predict ``split.test`` once.

In [ ]:
def run_job(cfg: dict[str, Any], split: Split, table: pd.DataFrame, feature: str,
            target: str, model_name: str) -> tuple[pd.DataFrame, dict[str, Any]]:
    """Tune one model by blocked CV inside ``split.train``, refit, predict ``split.test`` once.

    ``table`` is the row table of the task (IV rows for log10_I, curves otherwise).
    Returns the test predictions (one row per test row) and a metadata dict.
    """
    d = _task_data(cfg, split, table, feature, target)
    t0 = time.perf_counter()
    model, params, cv_mse = _tuned(cfg, model_name, target, d)
    mean, std = _predict(model, d["X_test"], model_name)
    pred = d["test_keys"].assign(y_true=d["y_test"], y_pred=mean, y_std=std)
    meta = {"cv_mse": cv_mse, "best_params": json.dumps({k: str(v) for k, v in params.items()}),
            "fit_seconds": round(time.perf_counter() - t0, 2)}
    return pred, meta

### `_task_data` (function)
Everything one job needs: X/y for train and test, blocked-CV fold indices inside the training pool, the gate column (None for curve targets) and test row keys.

In [ ]:
def _task_data(cfg: dict[str, Any], split: Split, table: pd.DataFrame, feature: str,
               target: str) -> dict[str, Any]:
    """Everything one job needs: X/y for train and test, blocked-CV fold indices inside
    the training pool, the gate column (None for curve targets) and test row keys."""
    gate = cfg["data"]["gate_column"] if target == cfg["experiments"]["point_target"] else None
    train, test = (t.reset_index(drop=True) for t in split.apply(table))
    keys = test[["T_K", "vac_pct"] + ([gate] if gate else [])]
    return {
        "gate": gate,
        "X_train": features(train, feature, gate), "y_train": train[target].to_numpy(),
        "X_test": features(test, feature, gate), "y_test": test[target].to_numpy(),
        "cv": fold_indices(train, blocked_cv_folds(train, cfg, split.cv_axis)),
        "test_keys": keys.rename(columns={gate: "gate_V"} if gate else {}),
    }

### `_log_memory` (function)
Append this process's memory after a job: current resident set and its peak so far (peak working set on Windows; NaN where the OS does not report it). One row per job, so growth over a run is visible.

In [ ]:
def _log_memory(log: Path, job: dict[str, str], seconds: float) -> None:
    """Append this process's memory after a job: current resident set and its peak so far
    (peak working set on Windows; NaN where the OS does not report it). One row per job, so
    growth over a run is visible."""
    info = psutil.Process().memory_info()
    peak = getattr(info, "peak_wset", float("nan"))
    row = pd.DataFrame([{"finished": time.strftime("%Y-%m-%d %H:%M:%S"), "pid": os.getpid(), **job,
                         "job_seconds": seconds, "rss_MB": round(info.rss / 2**20, 1),
                         "peak_MB": round(peak / 2**20, 1)}])
    row.to_csv(log, mode="a", header=not log.exists(), index=False)

### `_cached` (function)
Return a job's (predictions, metadata) from disk, or run ``compute`` and store it. Makes long runs resumable: an interrupted run skips every finished job. ``tag`` adds job-specific settings to the file name (e.g. the log10_I epoch limit). Newly computed jobs are recorded in ``memory_log`` if given.

In [ ]:
def _cached(cache: Path, job: dict[str, str], compute: Callable[[], tuple[pd.DataFrame, dict]],
            tag: str = "", memory_log: Path | None = None) -> tuple[pd.DataFrame, dict[str, Any]]:
    """Return a job's (predictions, metadata) from disk, or run ``compute`` and store it.
    Makes long runs resumable: an interrupted run skips every finished job. ``tag`` adds
    job-specific settings to the file name (e.g. the log10_I epoch limit). Newly computed
    jobs are recorded in ``memory_log`` if given."""
    path = cache / ("__".join([*job.values(), *([tag] if tag else [])]) + ".pkl")
    if path.exists():
        return pd.read_pickle(path)
    pred, meta = compute()
    pred = pred.assign(**job)
    pd.to_pickle((pred, meta), path)
    if memory_log is not None:
        _log_memory(memory_log, job, meta["fit_seconds"])
    print(f"  {job} cv_mse={meta['cv_mse']:.4g} ({meta['fit_seconds']} s)", flush=True)
    return pred, meta

### `_jobs` (function)
Every (protocol, vacancy feature, target, model) combination, in run order.

In [ ]:
def _jobs(cfg: dict[str, Any], splits: list[Split]):
    """Every (protocol, vacancy feature, target, model) combination, in run order."""
    e = cfg["experiments"]
    tasks = [(e["point_target"], f) for f in e["point_vacancy_features"]] + [
        (t, f) for t in e["curve_targets"] for f in cfg["inputs"]["vacancy_features"]]
    for split in splits:
        for target, feature in tasks:
            for model in MODELS:
                yield split, {"protocol": split.name, "feature": feature, "target": target, "model": model}

### `_score` (function)
Metrics per job plus skill against the best naive baseline of the same task.

In [ ]:
def _score(pred: pd.DataFrame, meta: pd.DataFrame) -> pd.DataFrame:
    """Metrics per job plus skill against the best naive baseline of the same task."""
    task = ["protocol", "feature", "target"]
    rows = [{**dict(zip([*task, "model"], k)), **regression_metrics(g.y_true, g.y_pred)}
            for k, g in pred.groupby([*task, "model"], sort=False)]
    m = pd.DataFrame(rows).merge(meta, on=[*task, "model"])
    best_naive = m[m.model.isin(NAIVE_MODELS)].groupby(task).MSE.min().rename("MSE_best_naive")
    m = m.merge(best_naive, left_on=task, right_index=True)
    return m.assign(skill=[skill_score(a, b) for a, b in zip(m.MSE, m.MSE_best_naive)])

### `step_classical` (function)
Baselines, ridge, polynomial ridge, boosting and GP on every protocol and target.

In [ ]:
def step_classical(cfg: dict[str, Any], out_dir: Path) -> list[Path]:
    """Baselines, ridge, polynomial ridge, boosting and GP on every protocol and target."""
    ds = load_dataset(cfg)
    curves = curve_targets(ds.metrics_pooled(), ds.iv_pooled(), cfg)
    iv = ds.iv_pooled().reset_index(drop=True)
    cache = repo_path(cfg, "cache_dir") / f"classical_{_config_hash(cfg, 'classical')}"
    cache.mkdir(parents=True, exist_ok=True)
    preds, metas = [], []
    for split, job in _jobs(cfg, all_protocols(curves, cfg)):
        table = iv if job["target"] == cfg["experiments"]["point_target"] else curves
        pred, meta = _cached(cache, job, partial(run_job, cfg, split, table, job["feature"], job["target"], job["model"]),
                             tag=split.fingerprint())
        preds.append(pred)
        metas.append({**job, **meta})
    pred = pd.concat(preds, ignore_index=True)
    paths = [out_dir / "predictions_classical.csv.gz", out_dir / "metrics_classical.csv"]
    pred.to_csv(paths[0], index=False, compression=GZIP)
    _score(pred, pd.DataFrame(metas)).to_csv(paths[1], index=False)
    return paths

### `_fit_seed` (function)
Refit the tuned MLP with one initialisation seed; return test predictions, best epoch.

In [ ]:
def _fit_seed(best: Any, seed: int, d: dict[str, Any]) -> tuple[np.ndarray, int]:
    """Refit the tuned MLP with one initialisation seed; return test predictions, best epoch."""
    model = clone(best).set_params(seed=seed).fit(d["X_train"], d["y_train"])
    return np.ravel(model.predict(d["X_test"])), model.best_epoch_, model.stop_epoch_   # (n,20) -> rows

### `_curve_features` (function)
(n_curves, 2) X = [T_K, vacancy feature] for curve keys, in the order of ``keys``.

In [ ]:
def _curve_features(keys: pd.DataFrame, per_curve: pd.DataFrame, feature: str) -> np.ndarray:
    """(n_curves, 2) X = [T_K, vacancy feature] for curve keys, in the order of ``keys``."""
    return features(keys.merge(per_curve, on=["T_K", "vac_pct"], how="left", validate="one_to_one"),
                    feature, None)

### `_curve_output_data` (function)
log10_I as one row per curve: X = (T_K, vacancy), y = the curve's 20 log10_I values.

In [ ]:
def _curve_output_data(cfg: dict[str, Any], split: Split, iv: pd.DataFrame, feature: str
                       ) -> dict[str, Any]:
    """log10_I as one row per curve: X = (T_K, vacancy), y = the curve's 20 log10_I values.

    Same keys as ``_task_data``; ``y_test`` and ``test_keys`` are flattened back to one row
    per (curve, gate) in curve-major order, so scoring matches the per-point models exactly.
    """
    train, test = (t.reset_index(drop=True) for t in split.apply(iv))
    cols = list(dict.fromkeys(["T_K", "vac_pct", feature]))   # feature may itself be vac_pct
    per_curve = iv[cols].drop_duplicates(["T_K", "vac_pct"])
    tr_keys, _, Y_train = curve_matrix(train, cfg, cfg["experiments"]["point_target"])
    te_keys, gate_V, Y_test = curve_matrix(test, cfg, cfg["experiments"]["point_target"])
    n_gate = len(gate_V)
    keys = pd.DataFrame({"T_K": np.repeat(te_keys["T_K"].to_numpy(), n_gate),
                         "vac_pct": np.repeat(te_keys["vac_pct"].to_numpy(), n_gate),
                         "gate_V": np.tile(gate_V, len(te_keys))})
    return {"gate": None, "X_train": _curve_features(tr_keys, per_curve, feature), "y_train": Y_train,
            "X_test": _curve_features(te_keys, per_curve, feature), "y_test": Y_test.ravel(),
            "cv": fold_indices(tr_keys, blocked_cv_folds(tr_keys, cfg, split.cv_axis)), "test_keys": keys}

### `_mlp_task_data` (function)
Curve-output layout for log10_I (``experiments.mlp_iv_mode``), the usual one otherwise.

In [ ]:
def _mlp_task_data(cfg: dict[str, Any], split: Split, table: pd.DataFrame, target: str
                   ) -> dict[str, Any]:
    """Curve-output layout for log10_I (``experiments.mlp_iv_mode``), the usual one otherwise."""
    feature = cfg["experiments"]["mlp_vacancy_feature"]
    if target == cfg["experiments"]["point_target"]:
        if cfg["experiments"]["mlp_iv_mode"] != "curve_output":
            raise ValueError(f"unknown mlp_iv_mode {cfg['experiments']['mlp_iv_mode']!r}")
        return _curve_output_data(cfg, split, table, feature)
    return _task_data(cfg, split, table, feature, target)

### `_iv_tag` (function)
Cache-name tag for settings that apply to the log10_I MLP only ("" for other targets).

In [ ]:
def _iv_tag(cfg: dict[str, Any], target: str) -> str:
    """Cache-name tag for settings that apply to the log10_I MLP only ("" for other targets)."""
    e = cfg["experiments"]
    return f"maxep{e['mlp_iv_max_epochs']}" if target == e["point_target"] else ""

### `run_mlp_job` (function)
Tune the MLP by blocked CV with the first seed, then refit with every seed in ``models.mlp.seeds`` (in parallel). Returns predictions (one block per seed) and metadata.

In [ ]:
def run_mlp_job(cfg: dict[str, Any], split: Split, table: pd.DataFrame, target: str
                ) -> tuple[pd.DataFrame, dict[str, Any]]:
    """Tune the MLP by blocked CV with the first seed, then refit with every seed in
    ``models.mlp.seeds`` (in parallel). Returns predictions (one block per seed) and metadata."""
    d = _mlp_task_data(cfg, split, table, target)
    t0 = time.perf_counter()
    best, params, cv_mse = _tuned(cfg, "mlp", target, d)
    seeds = cfg["models"]["mlp"]["seeds"]
    runs = Parallel(n_jobs=cfg["compute"]["n_jobs"])(delayed(_fit_seed)(best, s, d) for s in seeds)
    pred = pd.concat([d["test_keys"].assign(seed=s, y_true=d["y_test"], y_pred=p, y_std=np.nan)
                      for s, (p, _, _) in zip(seeds, runs)], ignore_index=True)
    meta = {"cv_mse": cv_mse, "best_params": json.dumps({k: str(v) for k, v in params.items()}),
            "best_epochs": json.dumps([int(b) for _, b, _ in runs]),
            "stop_epochs": json.dumps([int(st) for _, _, st in runs]),
            "max_epochs": int(best.max_epochs),
            "seeds_hit_epoch_cap": sum(int(st) == best.max_epochs - 1 for _, _, st in runs),
            "fit_seconds": round(time.perf_counter() - t0, 2)}
    return pred, meta

### `_mlp_score` (function)
Per-seed metrics and skill against the best naive baseline from the classical run.

In [ ]:
def _mlp_score(pred: pd.DataFrame, meta: pd.DataFrame, classical: pd.DataFrame) -> pd.DataFrame:
    """Per-seed metrics and skill against the best naive baseline from the classical run."""
    task = ["protocol", "feature", "target"]
    rows = [{**dict(zip([*task, "seed"], k)), **regression_metrics(g.y_true, g.y_pred)}
            for k, g in pred.groupby([*task, "seed"], sort=False)]
    naive = classical[classical.model.isin(NAIVE_MODELS)].groupby(task).MSE.min().rename("MSE_best_naive")
    m = pd.DataFrame(rows).merge(meta, on=task).merge(naive, left_on=task, right_index=True)
    return m.assign(model="mlp", skill=[skill_score(a, b) for a, b in zip(m.MSE, m.MSE_best_naive)])

### `_mlp_summary` (function)
MLP mean +/- std over seeds next to the best classical (non-naive) and best naive model.

In [ ]:
def _mlp_summary(per_seed: pd.DataFrame, classical: pd.DataFrame) -> pd.DataFrame:
    """MLP mean +/- std over seeds next to the best classical (non-naive) and best naive model."""
    task = ["protocol", "feature", "target"]
    s = per_seed.groupby(task).agg(mlp_MAE_mean=("MAE", "mean"), mlp_MAE_std=("MAE", "std"),
                                   mlp_skill_mean=("skill", "mean"), mlp_skill_std=("skill", "std"))
    c = classical.set_index(task).loc[s.index.unique()]
    best = (c[~c.model.isin(NAIVE_MODELS)].reset_index().sort_values("MAE").groupby(task).first()
            [["model", "MAE", "skill"]].add_prefix("best_classical_"))
    naive = c[c.model.isin(NAIVE_MODELS)].groupby(task).MAE.min().rename("best_naive_MAE")
    out = s.join(best).join(naive).reset_index()
    return out.assign(mlp_beats_best_classical=out.mlp_MAE_mean < out.best_classical_MAE)

### `_mlp_cfg` (function)
Copy of cfg with compute.n_jobs = compute.mlp_n_jobs (MLP fits need more memory).

In [ ]:
def _mlp_cfg(cfg: dict[str, Any]) -> dict[str, Any]:
    """Copy of cfg with compute.n_jobs = compute.mlp_n_jobs (MLP fits need more memory)."""
    return {**cfg, "compute": {**cfg["compute"], "n_jobs": cfg["compute"]["mlp_n_jobs"]}}

### `step_mlp` (function)
MLP on every protocol for ``experiments.mlp_targets``; needs the classical results. Uses ``compute.mlp_n_jobs`` parallel fits (MLP fits need more memory than GP fits).

In [ ]:
def step_mlp(cfg: dict[str, Any], out_dir: Path) -> list[Path]:
    """MLP on every protocol for ``experiments.mlp_targets``; needs the classical results.
    Uses ``compute.mlp_n_jobs`` parallel fits (MLP fits need more memory than GP fits)."""
    cfg = _mlp_cfg(cfg)
    ds = load_dataset(cfg)
    curves = curve_targets(ds.metrics_pooled(), ds.iv_pooled(), cfg)
    iv = ds.iv_pooled().reset_index(drop=True)
    classical = pd.read_csv(out_dir / "metrics_classical.csv")
    cache = repo_path(cfg, "cache_dir") / f"mlp_{_config_hash(cfg, 'mlp')}"
    cache.mkdir(parents=True, exist_ok=True)
    only = cfg.get("run", {}).get("protocols")        # --protocols: fill the cache for these only
    preds, metas = [], []
    for split in all_protocols(curves, cfg):
        if only and split.name not in only:
            continue
        for target in cfg["experiments"]["mlp_targets"]:
            job = {"protocol": split.name, "feature": cfg["experiments"]["mlp_vacancy_feature"], "target": target}
            table = iv if target == cfg["experiments"]["point_target"] else curves
            pred, meta = _cached(cache, job, partial(run_mlp_job, cfg, split, table, target),
                                 tag="_".join(filter(None, [split.fingerprint(), _iv_tag(cfg, target)])),
                                 memory_log=out_dir / "memory_log.csv")
            preds.append(pred)
            metas.append({**job, **meta})
    if only:
        print(f"  cached MLP jobs for {only}; run 'mlp' without --protocols to write the result files")
        return []
    pred = pd.concat(preds, ignore_index=True)
    per_seed = _mlp_score(pred, pd.DataFrame(metas), classical)
    paths = [out_dir / "predictions_mlp.csv.gz", out_dir / "metrics_mlp.csv", out_dir / "mlp_vs_classical.csv"]
    pred.to_csv(paths[0], index=False, compression=GZIP)
    per_seed.to_csv(paths[1], index=False)
    _mlp_summary(per_seed, classical).to_csv(paths[2], index=False)
    return paths

### `_pca_inputs` (function)
(curve table aligned to the curve matrix, gate grid, (798, 20) log10_I matrix).

In [ ]:
def _pca_inputs(cfg: dict[str, Any]) -> tuple[pd.DataFrame, np.ndarray, np.ndarray]:
    """(curve table aligned to the curve matrix, gate grid, (798, 20) log10_I matrix)."""
    ds = load_dataset(cfg)
    keys, gate_V, Y = curve_matrix(ds.iv_pooled(), cfg)
    curves = curve_targets(ds.metrics_pooled(), ds.iv_pooled(), cfg)
    aligned = keys.merge(curves, on=["T_K", "vac_pct"], how="left", validate="one_to_one")
    return aligned, gate_V, Y

### `step_unsupervised` (function)
PCA of the I-V curves and k-means / Gaussian-mixture clustering of the devices.

In [ ]:
def step_unsupervised(cfg: dict[str, Any], out_dir: Path) -> list[Path]:
    """PCA of the I-V curves and k-means / Gaussian-mixture clustering of the devices."""
    u, seed = cfg["unsupervised"], cfg["models"]["seed"]
    curves, gate_V, Y = _pca_inputs(cfg)
    pca = fit_pca(Y, u["pca_max_components"])
    n_keep = n_components_for(pca, u["cluster_variance_kept"])
    Z = pca.transform(Y)[:, :n_keep]
    classes = ion_ioff_classes(curves["log10_Ion_Ioff"], u["ion_ioff_class_edges"])
    scores = cluster_scores(Z, classes, u, seed).assign(n_pcs_used=n_keep)
    assign = curves[["T_K", "vac_pct", "log10_Ion_Ioff"]].assign(
        PC1=Z[:, 0], PC2=Z[:, 1], ion_ioff_class=np.array(u["class_names"])[classes])
    for method in ("kmeans", "gmm"):
        for k in u["k_range"]:
            assign[f"{method}_k{k}"] = cluster(Z, method, k, u, seed)[0]
    paths = [out_dir / n for n in ("pca_explained_variance.csv", "pca_components.csv",
                                   "cluster_scores.csv", "cluster_assignments.csv")]
    explained_variance_table(pca).to_csv(paths[0], index=False)
    component_table(pca, gate_V).to_csv(paths[1], index=False)
    scores.to_csv(paths[2], index=False)
    assign.to_csv(paths[3], index=False)
    return paths

### `_robustness_job` (function)
Tune and fit one model on clean data, then score it on noisy copies of the test inputs.

In [ ]:
def _robustness_job(cfg: dict[str, Any], split: Split, table: pd.DataFrame, target: str,
                    model_name: str) -> pd.DataFrame:
    """Tune and fit one model on clean data, then score it on noisy copies of the test inputs."""
    r = cfg["robustness"]
    run_cfg = _mlp_cfg(cfg) if model_name == "mlp" else cfg
    d = (_mlp_task_data(run_cfg, split, table, target) if model_name == "mlp"
         else _task_data(run_cfg, split, table, cfg["inputs"]["vacancy_features"][0], target))
    model = _tuned(run_cfg, model_name, target, d)[0]
    steps = input_steps(cfg, d["X_test"].shape[1])
    curve = noise_curve(model, d["X_test"], d["y_test"], steps, r["noise_fractions"], r["repeats"], r["seed"])
    return curve.assign(protocol=split.name, target=target, model=model_name)

### `step_robustness` (function)
Gaussian input-noise test (project proposal) for the models and protocols in config.

In [ ]:
def step_robustness(cfg: dict[str, Any], out_dir: Path) -> list[Path]:
    """Gaussian input-noise test (project proposal) for the models and protocols in config."""
    r = cfg["robustness"]
    ds = load_dataset(cfg)
    iv = ds.iv_pooled().reset_index(drop=True)
    curves = curve_targets(ds.metrics_pooled(), iv, cfg)
    splits = {s.name: s for s in all_protocols(curves, cfg)}
    parts = []
    for name in r["protocols"]:
        for target in r["targets"]:
            table = iv if target == cfg["experiments"]["point_target"] else curves
            for model_name in r["models"]:
                t0 = time.perf_counter()
                parts.append(_robustness_job(cfg, splits[name], table, target, model_name))
                print(f"  robustness {name} {target} {model_name} ({time.perf_counter() - t0:.0f} s)", flush=True)
    raw = pd.concat(parts, ignore_index=True)
    paths = [out_dir / "robustness_noise.csv", out_dir / "robustness_summary.csv"]
    raw.to_csv(paths[0], index=False)
    summarise(raw, ["protocol", "target", "model"]).to_csv(paths[1], index=False)
    return paths

### `_per_seed` (function)
Per-seed MAE of one MLP job, with the epoch each seed stopped at and whether it hit ``cap``.

In [ ]:
def _per_seed(pred: pd.DataFrame, meta: dict[str, Any], cap: int) -> pd.DataFrame:
    """Per-seed MAE of one MLP job, with the epoch each seed stopped at and whether it hit ``cap``."""
    stops, bests = json.loads(meta["stop_epochs"]), json.loads(meta["best_epochs"])
    rows = [{"seed": s, "MAE": regression_metrics(g.y_true, g.y_pred)["MAE"]} for s, g in pred.groupby("seed")]
    return pd.DataFrame(rows).assign(max_epochs=cap, stop_epoch=stops, best_epoch=bests,
                                     hit_cap=[st == cap - 1 for st in stops])

### `step_epoch_sensitivity` (function)
Re-run selected curve-level MLP jobs with a higher epoch limit; compare per seed with the main run (results/metrics_mlp.csv). Own cache folder and output: main results are untouched.

In [ ]:
def step_epoch_sensitivity(cfg: dict[str, Any], out_dir: Path) -> list[Path]:
    """Re-run selected curve-level MLP jobs with a higher epoch limit; compare per seed with the
    main run (results/metrics_mlp.csv). Own cache folder and output: main results are untouched."""
    e = cfg["epoch_sensitivity"]
    high = _mlp_cfg(cfg)
    high = {**high, "models": {**high["models"], "mlp": {**high["models"]["mlp"], "max_epochs": e["max_epochs"]}}}
    ds = load_dataset(cfg)
    curves = curve_targets(ds.metrics_pooled(), ds.iv_pooled(), cfg)
    splits = {s.name: s for s in all_protocols(curves, cfg)}
    cache = repo_path(cfg, "cache_dir") / f"mlp_epochsens_{_config_hash(high, 'mlp')}"
    cache.mkdir(parents=True, exist_ok=True)
    base = pd.read_csv(out_dir / "metrics_mlp.csv")
    rows = []
    for protocol, target in e["jobs"]:
        job = {"protocol": protocol, "feature": cfg["experiments"]["mlp_vacancy_feature"], "target": target}
        pred, meta = _cached(cache, job, partial(run_mlp_job, high, splits[protocol], curves, target),
                             tag=splits[protocol].fingerprint(), memory_log=out_dir / "memory_log.csv")
        old = base[(base.protocol == protocol) & (base.target == target)].sort_values("seed")
        old_meta = {k: old[k].iloc[0] for k in ("stop_epochs", "best_epochs")}
        rows.append(_per_seed(pred, meta, e["max_epochs"]).assign(protocol=protocol, target=target,
                                                                   fit_seconds=meta["fit_seconds"]))
        base_seed = old[["seed", "MAE"]].assign(max_epochs=cfg["models"]["mlp"]["max_epochs"],
                                                stop_epoch=json.loads(old_meta["stop_epochs"]),
                                                best_epoch=json.loads(old_meta["best_epochs"]))
        rows.append(base_seed.assign(hit_cap=base_seed.stop_epoch == cfg["models"]["mlp"]["max_epochs"] - 1,
                                     protocol=protocol, target=target, fit_seconds=old.fit_seconds.iloc[0]))
    path = out_dir / "epoch_sensitivity.csv"
    pd.concat(rows, ignore_index=True).to_csv(path, index=False)
    return [path]

### `_best_naive` (function)
Per (protocol, target): the naive baseline with the lower test MSE (the skill reference).

In [ ]:
def _best_naive(classical: pd.DataFrame) -> pd.Series:
    """Per (protocol, target): the naive baseline with the lower test MSE (the skill reference)."""
    naive = classical[classical.model.isin(NAIVE_MODELS)]
    return naive.loc[naive.groupby(["protocol", "target"]).MSE.idxmin()].set_index(["protocol", "target"]).model

### `step_uncertainty` (function)
Cluster-bootstrap CIs (whole vacancy levels) of every test MAE, paired differences vs the best naive baseline, and GP interval coverage. Reads saved predictions only; no refits.

In [ ]:
def step_uncertainty(cfg: dict[str, Any], out_dir: Path) -> list[Path]:
    """Cluster-bootstrap CIs (whole vacancy levels) of every test MAE, paired differences vs the
    best naive baseline, and GP interval coverage. Reads saved predictions only; no refits."""
    u, step = cfg["uncertainty"], cfg["splits"]["vac_step_pct"]
    classical = pd.read_csv(out_dir / "metrics_classical.csv")
    pred = pooled_predictions(pd.read_csv(out_dir / "predictions_classical.csv.gz"),
                              pd.read_csv(out_dir / "predictions_mlp.csv.gz"))
    pred = pred[pred.feature == cfg["inputs"]["vacancy_features"][0]]
    reference = _best_naive(classical)
    models = sorted(pred.model.unique())
    n_comp = len(reference) * (len(models) - 1)
    parts = [task_bootstrap(g, models, reference[k], u, step, n_comp).assign(protocol=k[0], target=k[1])
             for k, g in pred.groupby(["protocol", "target"])]
    gp = pred[pred.model == "gp"]
    cover = pd.DataFrame([{"protocol": k[0], "target": k[1], **gp_coverage(g)}
                          for k, g in gp.groupby(["protocol", "target"])])
    paths = [out_dir / "bootstrap_ci.csv", out_dir / "gp_coverage.csv"]
    pd.concat(parts, ignore_index=True).to_csv(paths[0], index=False)
    cover.assign(nominal_68=0.6827, nominal_95=0.95).to_csv(paths[1], index=False)
    return paths

In [ ]:
STEPS = {"targets": step_targets, "splits": step_splits, "classical": step_classical,
         "mlp": step_mlp, "unsupervised": step_unsupervised, "robustness": step_robustness,
         "epoch_sensitivity": step_epoch_sensitivity, "uncertainty": step_uncertainty}

## Figures (read results/*.csv, write figures/*.png)
*Source: `python/make_figures.py`*

Produce figures/*.png. Run from the repo root:  .\.venv\Scripts\python python\make_figures.py

Diagnostic figures (swing check, PCA) and the Stage 7 report figures. Result figures read
results/*.csv only; plotting primitives live in wte2ml.plots.

In [ ]:
from pathlib import Path
from typing import Any
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.interpolate import PchipInterpolator

### `_plot_one` (function)
One panel: raw points, falling branch, PCHIP fit, steepest-slope tangent.

In [ ]:
def _plot_one(ax, gate_V: np.ndarray, y: np.ndarray, title: str, swing: float, n_fine: int,
              tangent_half_width_V: float) -> None:
    """One panel: raw points, falling branch, PCHIP fit, steepest-slope tangent."""
    br = falling_branch(y)
    ax.plot(gate_V, y, "o", ms=3, color="0.6", label="NEGF points")
    ax.plot(gate_V[br], y[br], "o", ms=3, color="C0", label="falling branch")
    if br.stop >= 2:
        fine = np.linspace(gate_V[0], gate_V[br.stop - 1], n_fine)
        f = PchipInterpolator(gate_V[br], y[br])
        ax.plot(fine, f(fine), "-", lw=1, color="C0")
        if np.isfinite(swing):
            k = np.argmin(f.derivative()(fine))
            near = np.abs(fine - fine[k]) <= tangent_half_width_V
            tang = f(fine[k]) + f.derivative()(fine[k]) * (fine[near] - fine[k])
            ax.plot(fine[near], tang, "--", lw=1.0, color="C3", label="steepest tangent")
    ax.set_title(title, fontsize=7)
    ax.tick_params(labelsize=7)

### `figure_swing_check` (function)
Grid of sample curves (T x vacancy from config) showing how swing is extracted.

In [ ]:
def figure_swing_check(cfg: dict[str, Any], out_dir: Path) -> Path:
    """Grid of sample curves (T x vacancy from config) showing how swing is extracted."""
    iv = load_dataset(cfg).iv_pooled()
    keys, gate_V, Y = curve_matrix(iv, cfg)
    table = swing_table(iv, cfg).set_index(["T_K", "vac_pct"])
    sw = cfg["targets"]["swing"]
    Ts, vacs = sw["check_T_K"], sw["check_vac_pct"]
    fig, axes = plt.subplots(len(Ts), len(vacs), figsize=(3.2 * len(vacs), 2.6 * len(Ts)),
                             sharex=True, squeeze=False)
    lookup = {(t, v): i for i, (t, v) in enumerate(keys.itertuples(index=False))}
    for (r, T), (c, vac) in ((a, b) for a in enumerate(Ts) for b in enumerate(vacs)):
        s = table.loc[(T, vac)]
        title = (f"{T} K, {vac:.1f} %: S={s.swing_mV_per_dec:.1f} (raw {s.swing_raw_mV_per_dec:.0f},"
                 f" kTln10 {s.swing_boltzmann_mV_per_dec:.1f})")
        _plot_one(axes[r, c], gate_V, Y[lookup[(T, vac)]], title, s.swing_mV_per_dec,
                  sw["pchip_points"], sw["tangent_half_width_V"])
    for ax in axes[-1]:
        ax.set_xlabel("V_TG - V_BG (V)", fontsize=8)
    for ax in axes[:, 0]:
        ax.set_ylabel("log10 I (A)", fontsize=8)
    axes[0, 0].legend(fontsize=6)
    fig.suptitle("Swing extraction check (S in mV/dec; dashed = steepest PCHIP tangent)", fontsize=9)
    fig.tight_layout()
    path = out_dir / "swing_check.png"
    fig.savefig(path, dpi=150)
    plt.close(fig)
    return path

### `_plot_variance` (function)
Bars: share of variance per component; line: cumulative share.

In [ ]:
def _plot_variance(ax, ev: pd.DataFrame) -> None:
    """Bars: share of variance per component; line: cumulative share."""
    ax.bar(ev.component, ev.explained_variance_ratio, color="C0", label="per component")
    ax.plot(ev.component, ev.cumulative, "o-", color="C3", ms=3, label="cumulative")
    ax.set(xlabel="principal component", ylabel="share of variance", ylim=(0, 1.02),
           title="(a) PCA of log10 I-V curves")
    ax.legend(fontsize=7)

### `_plot_loadings` (function)
What PC1 and PC2 add to the mean curve, per gate voltage.

In [ ]:
def _plot_loadings(ax, comps: pd.DataFrame) -> None:
    """What PC1 and PC2 add to the mean curve, per gate voltage."""
    for c, style in ((1, "-"), (2, "--")):
        sub = comps[comps.component == c]
        ax.plot(sub.gate_V, sub.loading, style, label=f"PC{c} loading")
    ax.axhline(0, color="0.7", lw=0.8)
    ax.set(xlabel="V_TG - V_BG (V)", ylabel="loading (decades per unit score)",
           title="(b) component shapes")
    ax.legend(fontsize=7)

### `_plot_clusters` (function)
PC1-PC2 scores coloured by cluster, marker by Ion/Ioff class.

In [ ]:
def _plot_clusters(ax, assign: pd.DataFrame, column: str, class_names: list[str]) -> None:
    """PC1-PC2 scores coloured by cluster, marker by Ion/Ioff class."""
    markers = dict(zip(class_names, ("x", "o", "^")))
    for cls, m in markers.items():
        sub = assign[assign.ion_ioff_class == cls]
        ax.scatter(sub.PC1, sub.PC2, c=sub[column], cmap="viridis", vmin=0,
                   vmax=assign[column].max(), marker=m, s=14, lw=0.8, label=cls)
    ax.set(xlabel="PC1 score", ylabel="PC2 score", title=f"(c) {column} (colour) vs Ion/Ioff class (marker)")
    leg = ax.legend(fontsize=7, title="log10 Ion/Ioff class", title_fontsize=7)
    for handle in leg.legend_handles:    # marker shape = class; colour belongs to the clusters
        handle.set_color("0.35")

### `figure_pca_clusters` (function)
Three panels from results/pca_*.csv and cluster_assignments.csv (Stage 6).

In [ ]:
def figure_pca_clusters(cfg: dict[str, Any], out_dir: Path, column: str = "kmeans_k3") -> Path:
    """Three panels from results/pca_*.csv and cluster_assignments.csv (Stage 6)."""
    res = repo_path(cfg, "results_dir")
    fig, axes = plt.subplots(1, 3, figsize=(15, 4.2))
    _plot_variance(axes[0], pd.read_csv(res / "pca_explained_variance.csv"))
    _plot_loadings(axes[1], pd.read_csv(res / "pca_components.csv"))
    _plot_clusters(axes[2], pd.read_csv(res / "cluster_assignments.csv"), column,
                   cfg["unsupervised"]["class_names"])
    fig.tight_layout()
    path = out_dir / "pca_clusters.png"
    fig.savefig(path, dpi=150)
    plt.close(fig)
    return path

In [ ]:
PROTOCOLS = ["supplied", "temperature_forward", "vacancy_block", "vacancy_extrapolation"]
FIGURE_MODELS = ["naive_vac_mean", "naive_nn", "ridge", "poly_ridge", "boosting", "gp", "mlp"]

### `_save` (function)

In [ ]:
def _save(fig, out_dir: Path, name: str) -> Path:
    fig.tight_layout()
    path = out_dir / name
    fig.savefig(path, dpi=150)
    plt.close(fig)
    return path

### `load_predictions` (function)
All test predictions (MLP averaged over seeds), see evaluate.pooled_predictions.

In [ ]:
def load_predictions(res: Path) -> pd.DataFrame:
    """All test predictions (MLP averaged over seeds), see evaluate.pooled_predictions."""
    return pooled_predictions(pd.read_csv(res / "predictions_classical.csv.gz"),
                              pd.read_csv(res / "predictions_mlp.csv.gz"))

### `figure_ratio_heatmap` (function)
log10 Ion/Ioff (supplied end-point definition) over the full T x vacancy grid.

In [ ]:
def figure_ratio_heatmap(res: Path, out_dir: Path) -> Path:
    """log10 Ion/Ioff (supplied end-point definition) over the full T x vacancy grid."""
    curves = pd.read_csv(res / "curve_targets.csv")
    fig, ax = plt.subplots(figsize=(7, 4.5))
    im = heatmap(ax, curves, "log10_Ion_Ioff", "log10 Ion/Ioff (all 798 curves)")
    fig.colorbar(im, ax=ax, label="log10 Ion/Ioff (decades)")
    return _save(fig, out_dir, "ion_ioff_heatmap.png")

### `figure_pred_vs_true` (function)
Grid: rows = protocols, columns = models; predicted vs true for one target.

In [ ]:
def figure_pred_vs_true(pred: pd.DataFrame, target: str, out_dir: Path) -> Path:
    """Grid: rows = protocols, columns = models; predicted vs true for one target."""
    sub = pred[pred.target == target]
    lims = (float(sub.y_true.min()) - 0.5, float(sub.y_true.max()) + 0.5)
    fig, axes = plt.subplots(len(PROTOCOLS), len(FIGURE_MODELS), figsize=(2.1 * len(FIGURE_MODELS), 2.1 * len(PROTOCOLS)))
    for r, protocol in enumerate(PROTOCOLS):
        for c, model in enumerate(FIGURE_MODELS):
            cell = sub[(sub.protocol == protocol) & (sub.model == model)]
            predicted_vs_true(axes[r, c], cell, f"{protocol}\n{model}", lims)
    fig.supxlabel(f"true {target}"); fig.supylabel(f"predicted {target}")
    return _save(fig, out_dir, f"pred_vs_true_{target}.png")

### `figure_error_maps` (function)
|error| of each test curve over the T x vacancy grid; rows = protocols, cols = models.

In [ ]:
def figure_error_maps(pred: pd.DataFrame, target: str, models: list[str], out_dir: Path) -> Path:
    """|error| of each test curve over the T x vacancy grid; rows = protocols, cols = models."""
    sub = pred[pred.target == target].assign(abs_err=lambda d: (d.y_pred - d.y_true).abs())
    vmax = float(sub.abs_err.quantile(0.98))
    fig, axes = plt.subplots(len(PROTOCOLS), len(models), figsize=(3.4 * len(models), 2.9 * len(PROTOCOLS)),
                             layout="constrained")
    for r, protocol in enumerate(PROTOCOLS):
        for c, model in enumerate(models):
            cell = sub[(sub.protocol == protocol) & (sub.model == model)]
            im = heatmap(axes[r, c], cell, "abs_err", f"{protocol}: {model}", "magma", 0, vmax)
            axes[r, c].title.set_fontsize(8)
    fig.colorbar(im, ax=axes, label=f"|error| in {target} (decades)", shrink=0.6)
    path = out_dir / f"error_map_{target}.png"
    fig.savefig(path, dpi=150)
    plt.close(fig)
    return path

### `figure_error_profiles` (function)
MAE vs vacancy (top) and vs temperature (bottom), one column per protocol.

In [ ]:
def figure_error_profiles(pred: pd.DataFrame, target: str, out_dir: Path) -> Path:
    """MAE vs vacancy (top) and vs temperature (bottom), one column per protocol."""
    sub = pred[pred.target == target]
    fig, axes = plt.subplots(2, len(PROTOCOLS), figsize=(4 * len(PROTOCOLS), 6.5))
    for c, protocol in enumerate(PROTOCOLS):
        cell = sub[sub.protocol == protocol]
        error_profile(axes[0, c], cell, "vac_pct", FIGURE_MODELS, f"{protocol}: error vs vacancy")
        error_profile(axes[1, c], cell, "T_K", FIGURE_MODELS, f"{protocol}: error vs temperature")
    axes[0, 0].legend(fontsize=6)
    return _save(fig, out_dir, f"error_profiles_{target}.png")

### `figure_iv_samples` (function)
Sample I-V curves (rows = protocols) with GP mean +/- 2 std, boosting and MLP.

In [ ]:
def figure_iv_samples(pred: pd.DataFrame, cfg: dict[str, Any], out_dir: Path) -> Path:
    """Sample I-V curves (rows = protocols) with GP mean +/- 2 std, boosting and MLP."""
    iv = pred[pred.target == cfg["experiments"]["point_target"]]
    fig, axes = plt.subplots(len(PROTOCOLS), 3, figsize=(3.6 * 3, 2.8 * len(PROTOCOLS)))
    for r, protocol in enumerate(PROTOCOLS):
        cell = iv[iv.protocol == protocol]
        T = cfg["figures"]["iv_sample_T_K"][protocol]
        levels = np.sort(cell[cell.T_K == T].vac_pct.unique())
        for c, vac in enumerate(levels[[0, len(levels) // 2, -1]]):   # lowest, middle, highest test level
            curve = cell[(cell.T_K == T) & np.isclose(cell.vac_pct, vac)]
            iv_with_bands(axes[r, c], curve, ["gp", "boosting", "mlp"], f"{protocol}: {T} K, {vac:.1f} %")
    axes[0, 0].legend(fontsize=6)
    return _save(fig, out_dir, "iv_samples_gp_bands.png")

### `figure_robustness` (function)
Test MAE vs Gaussian input-noise level, one panel per protocol.

In [ ]:
def figure_robustness(res: Path, out_dir: Path) -> Path:
    """Test MAE vs Gaussian input-noise level, one panel per protocol."""
    s = pd.read_csv(res / "robustness_summary.csv")
    protocols = list(dict.fromkeys(s.protocol))
    fig, axes = plt.subplots(1, len(protocols), figsize=(5.5 * len(protocols), 4), squeeze=False)
    for ax, protocol in zip(axes[0], protocols):
        robustness_lines(ax, s[s.protocol == protocol], f"{protocol}: {s.target.iloc[0]}")
    axes[0, 0].legend(fontsize=7)
    return _save(fig, out_dir, "robustness_noise.png")

### `figure_comparison_table` (function)
Model x protocol MAE table for one target as an image (best per protocol in bold).

In [ ]:
def figure_comparison_table(table: pd.DataFrame, target: str, out_dir: Path) -> Path:
    """Model x protocol MAE table for one target as an image (best per protocol in bold)."""
    wide = table[table.target == target].pivot(index="model", columns="protocol", values="MAE")
    wide = wide.reindex(index=FIGURE_MODELS, columns=PROTOCOLS)
    fig, ax = plt.subplots(figsize=(8, 0.4 * len(FIGURE_MODELS) + 1))
    ax.axis("off")
    t = ax.table(cellText=wide.map(lambda v: f"{v:.3f}").to_numpy(), rowLabels=wide.index,
                 colLabels=wide.columns, loc="center")
    for c, protocol in enumerate(PROTOCOLS):
        t[(int(np.argmin(wide[protocol].to_numpy())) + 1, c)].set_text_props(fontweight="bold")
    ax.set_title(f"MAE (decades), {target}; bold = best per protocol", fontsize=9)
    return _save(fig, out_dir, f"model_comparison_{target}.png")

### `report_figures` (function)
Every Stage 7 figure plus results/model_comparison.csv.

In [ ]:
def report_figures(cfg: dict[str, Any], out_dir: Path) -> list[Path]:
    """Every Stage 7 figure plus results/model_comparison.csv."""
    res = repo_path(cfg, "results_dir")
    feature = cfg["inputs"]["vacancy_features"][0]          # never mix vacancy encodings in a figure
    table = comparison_table(pd.read_csv(res / "metrics_classical.csv"), pd.read_csv(res / "metrics_mlp.csv"))
    table = table[table.feature == feature]
    table.to_csv(res / "model_comparison.csv", index=False)
    pred = load_predictions(res)
    pred = pred[pred.feature == feature]
    head = cfg["figures"]["headline_target"]
    paths = [figure_ratio_heatmap(res, out_dir), figure_iv_samples(pred, cfg, out_dir),
             figure_robustness(res, out_dir)]
    for target in (head, cfg["experiments"]["point_target"]):
        paths += [figure_pred_vs_true(pred, target, out_dir), figure_error_profiles(pred, target, out_dir),
                  figure_comparison_table(table, target, out_dir),
                  figure_error_maps(pred, target, cfg["figures"]["error_map_models"], out_dir)]
    return paths

## Run the pipeline
Each step writes its `results/*.csv`; times are appended to `results/timings_log.csv`. Steps listed in `REUSE_GITHUB_RESULTS` download the saved results instead.

In [ ]:
cfg = {**load_config(), "run": {"protocols": None}}
out_dir = repo_path(cfg, "results_dir"); out_dir.mkdir(parents=True, exist_ok=True)
repo_path(cfg, "figures_dir").mkdir(parents=True, exist_ok=True)
timings = []
for step in RUN_STEPS:
    t0 = time.perf_counter()
    if step in REUSE_GITHUB_RESULTS:
        for f in REUSABLE[step]:
            download("results/" + f)
        how = "downloaded"
    else:
        for p in STEPS[step](cfg, out_dir):
            print("  wrote", p.name, flush=True)
        how = "computed"
    timings.append({"step": step, "how": how, "seconds": round(time.perf_counter() - t0, 1)})
    print(f"{step}: {how} in {timings[-1]['seconds']} s", flush=True)
import pandas as pd
log = out_dir / "timings_log.csv"          # appended, so every run's step times stay on record
pd.DataFrame(timings).assign(finished=time.strftime("%Y-%m-%d %H:%M:%S"), n_jobs=N_JOBS).to_csv(
    log, mode="a", header=not log.exists(), index=False)
pd.DataFrame(timings)

## Make every figure

In [ ]:
import matplotlib.pyplot as plt
from IPython.display import Image, display
fig_dir = repo_path(cfg, "figures_dir")
paths = [figure_swing_check(cfg, fig_dir), figure_pca_clusters(cfg, fig_dir), *report_figures(cfg, fig_dir)]
plt.close("all")
for p in paths:
    print(p.name); display(Image(str(p)))

## Key result tables

In [ ]:
comparison = pd.read_csv(out_dir / "model_comparison.csv")
comparison[comparison.target == "log10_Ion_Ioff"].pivot(index="model", columns="protocol", values="MAE").round(3)

In [ ]:
boot = pd.read_csv(out_dir / "bootstrap_ci.csv")
best = comparison[comparison["rank"] == 1][["protocol", "target", "model"]]
best.merge(boot, on=["protocol", "target", "model"])[["target", "protocol", "model", "n_levels", "MAE", "MAE_lo", "MAE_hi"]].round(3)

In [ ]:
pd.read_csv(out_dir / "gp_coverage.csv")[["protocol", "target", "coverage_68", "coverage_95"]].round(3)

In [ ]:
pd.read_csv(out_dir / "robustness_summary.csv").pivot_table(index=["protocol", "model"], columns="noise_fraction", values="MAE_mean").round(3)

In [ ]:
pd.read_csv(out_dir / "timings_log.csv")